# Step 0 — Setup

In [41]:
import os, pandas as pd

# Environment
CDR = os.environ["WORKSPACE_CDR"]

# Helper: BigQuery runner
def bq(sql): 
    return pd.read_gbq(sql, dialect="standard")

In [42]:
#Step 1 — Cohort (index date)
#Starts with surgical operation concept IDs
#Uses the concept hierarchy to find ALL related surgical concepts (descendants)
#CTE: Finds all events (visits, procedures) that match the surgical concepts
#Gets person_id and the event date (entry_date)
#Groups by person_id
#Takes the FIRST (MIN) surgery event date as the "index date" (baseline)
#This is the date when the surgery occurred

In [43]:
sql_index_surgery = f"""
WITH surgery_event_concepts AS (
  SELECT DISTINCT c.concept_id
  FROM `{CDR}.cb_criteria` c
  JOIN (
    SELECT CAST(cr.id AS STRING) AS id
    FROM `{CDR}.cb_criteria` cr
    WHERE concept_id IN (4213287, 42536436, 43531648, 42536435, 4058591,  # knee arthroplasty
                         4203771, 37017275, 42536396, 36674771, 42539518, 40481495,  # hip arthroplasty
                         4177164, 607983, 4174132, 36714117, 4312605, 40490415,  # fusion of lumbar spine
                         4163971, 4242997,  # Cholecystectomy
                         4127887, 4127886,  # Hysterectomy
                         4243973, 4198190,  # Appendectomy
                         4033396, 4195307,  # repair of inguinal hernia
                         4079713, 40481893, 4225427, 4195151,  # colectomy
                         4336464,  # Coronary artery bypass graft
                         4000882, 4073393,  # Lung excision
                         4209151)  # Repair of shoulder
      AND full_text LIKE '%_rank1]%'
  ) a
  ON (c.path LIKE CONCAT('%.', a.id, '.%')
   OR c.path LIKE CONCAT('%.', a.id)
   OR c.path LIKE CONCAT(a.id, '.%')
   OR c.path = a.id)
  WHERE c.is_standard = 1 AND c.is_selectable = 1
),
surgery_events AS (
  SELECT person_id, entry_date
  FROM `{CDR}.cb_search_all_events`
  WHERE concept_id IN (SELECT concept_id FROM surgery_event_concepts)
)
SELECT person_id, MIN(entry_date) AS index_date
FROM surgery_events
GROUP BY person_id
"""

surgery_index_df = bq(sql_index_surgery)
surgery_index_df["index_date"] = pd.to_datetime(surgery_index_df["index_date"]).dt.date
print("Cohort:", surgery_index_df.shape, "| persons:", surgery_index_df.person_id.nunique())

Cohort: (17101, 2) | persons: 17101


# Treatment(Intervention) List:

In [44]:
# Step 2 — Enhanced Treatment Definition (OPIOID drugs)
# POST-SURGERY ONLY + Cumulative Exposure + Time-Dependent Features

sql_opioid_enhanced = f"""
WITH cohort AS ({sql_index_surgery}),

-- 1. Identify TRUE opioid INGREDIENT concepts (RxNorm Ingredients)
opioid_ingredients AS (
  SELECT concept_id
  FROM `{CDR}.concept`
  WHERE concept_class_id = 'Ingredient'
    AND vocabulary_id = 'RxNorm'
    AND LOWER(concept_name) IN (
      'morphine',
      'oxycodone',
      'hydromorphone',
      'fentanyl',
      'codeine',
      'tramadol',
      'oxymorphone',
      'buprenorphine',
      'meperidine',
      'methadone'
    )
),

-- 2. Expand to descendant ingredients
opioid_ingredient_descendants AS (
  SELECT DISTINCT ca.descendant_concept_id AS ingredient_concept_id
  FROM `{CDR}.concept_ancestor` ca
  JOIN opioid_ingredients oi
    ON ca.ancestor_concept_id = oi.concept_id
),

-- 3. Find all DRUGS that contain ≥1 opioid ingredient
opioid_drugs AS (
  SELECT DISTINCT
    ds.drug_concept_id
  FROM `{CDR}.drug_strength` ds
  JOIN opioid_ingredient_descendants oid
    ON ds.ingredient_concept_id = oid.ingredient_concept_id
),

-- 4. Enhanced drug exposures with timing and cumulative features (POST-SURGERY ONLY)
opioid_exposures_detailed AS (
  SELECT
    de.person_id,
    de.drug_concept_id,
    DATE(de.drug_exposure_start_date) AS exposure_start_date,
    DATE(de.drug_exposure_end_date) AS exposure_end_date,
    c.index_date,
    -- Calculate days from surgery
    DATE_DIFF(DATE(de.drug_exposure_start_date), DATE(c.index_date), DAY) AS days_from_surgery,
    -- Calculate exposure duration (use days_supply if available, else estimate from dates)
    COALESCE(
      de.days_supply,
      DATE_DIFF(
        COALESCE(DATE(de.drug_exposure_end_date), DATE(de.drug_exposure_start_date)),
        DATE(de.drug_exposure_start_date),
        DAY
      ) + 1
    ) AS exposure_duration,
    -- Quantity (if available)
    de.quantity,
    -- Early exposure flag (first 3 days post-surgery - critical window)
    CASE 
      WHEN DATE_DIFF(DATE(de.drug_exposure_start_date), DATE(c.index_date), DAY) <= 3 
      THEN 1 ELSE 0 
    END AS early_exposure
  FROM `{CDR}.drug_exposure` de
  JOIN opioid_drugs od
    ON de.drug_concept_id = od.drug_concept_id
  JOIN cohort c
    ON de.person_id = c.person_id
  WHERE DATE(de.drug_exposure_start_date) >= DATE(c.index_date)
    AND DATE_DIFF(DATE(de.drug_exposure_start_date), DATE(c.index_date), DAY) <= 90
),

-- 5. Aggregate cumulative exposure per person-drug
opioid_exposures_aggregated AS (
  SELECT
    person_id,
    drug_concept_id,
    -- Binary: has drug (any exposure)
    MAX(1) AS has_drug,
    -- Cumulative days of exposure
    SUM(COALESCE(exposure_duration, 1)) AS total_days_exposed,
    -- Number of distinct exposures
    COUNT(*) AS n_exposures,
    -- Early exposure flag (any exposure in first 3 days)
    MAX(early_exposure) AS has_early_exposure,
    -- First exposure day (days from surgery)
    MIN(days_from_surgery) AS first_exposure_day,
    -- Total quantity (if available)
    SUM(COALESCE(quantity, 0)) AS total_quantity
  FROM opioid_exposures_detailed
  GROUP BY person_id, drug_concept_id
),

-- 6. Remove rare drugs
opioid_drug_counts AS (
  SELECT
    drug_concept_id,
    COUNT(DISTINCT person_id) AS n_patients
  FROM opioid_exposures_aggregated
  GROUP BY drug_concept_id
  HAVING n_patients >= 20
)

-- 7. Return detailed exposure data
SELECT
  oea.person_id,
  oea.drug_concept_id,
  oea.has_drug,
  oea.total_days_exposed,
  oea.n_exposures,
  oea.has_early_exposure,
  oea.first_exposure_day,
  oea.total_quantity
FROM opioid_exposures_aggregated oea
WHERE oea.drug_concept_id IN (SELECT drug_concept_id FROM opioid_drug_counts)
"""

opioid_enhanced = bq(sql_opioid_enhanced)

print(f"Enhanced opioid exposure data: {opioid_enhanced.shape}")
print(f"Unique drugs: {opioid_enhanced['drug_concept_id'].nunique()}")
print(f"Unique patients: {opioid_enhanced['person_id'].nunique()}")

Enhanced opioid exposure data: (39552, 8)
Unique drugs: 123
Unique patients: 10396


In [45]:
# Get all unique drug concept IDs from opioid_enhanced
drug_concept_ids = sorted(opioid_enhanced['drug_concept_id'].unique())
print(f"Found {len(drug_concept_ids)} unique opioid drugs with ≥20 patients")

# Get drug names
all_drug_names = {}
batch_size = 100
for i in range(0, len(drug_concept_ids), batch_size):
    batch = drug_concept_ids[i:i+batch_size]
    sql_drug_names = f"""
    SELECT concept_id, concept_name
    FROM `{CDR}.concept`
    WHERE concept_id IN ({','.join(map(str, batch))})
    """
    batch_names = bq(sql_drug_names)
    all_drug_names.update(dict(zip(batch_names['concept_id'], batch_names['concept_name'])))

# Initialize tx
if 'tx' not in locals():
    tx = surgery_index_df.copy()

# Create binary treatment columns from opioid_enhanced
for drug_id in drug_concept_ids:
    col_name = f"drug_{drug_id}"
    tx[col_name] = 0
    # Use opioid_enhanced instead of opioid_all_drugs
    patients_with_drug = opioid_enhanced[
        opioid_enhanced['drug_concept_id'] == drug_id
    ]['person_id'].unique()
    tx.loc[tx['person_id'].isin(patients_with_drug), col_name] = 1
    tx[col_name] = tx[col_name].astype("int8")

Found 123 unique opioid drugs with ≥20 patients


# Outcomes

In [46]:
# Step 4 — Opioid-related outcome (using provided concepts)
# CRITICAL FIX: Overdose must occur AFTER first opioid drug exposure

sql_overdose = f"""
WITH cohort AS ({sql_index_surgery}),

-- 1. Use PROVIDED opioid-related condition concepts
overdose_concepts AS (
  SELECT concept_id
  FROM `{CDR}.concept`
  WHERE concept_id IN (438130, 438120, 434016)
),

-- 2. Get first opioid drug exposure date per person (from Step 2 logic)
opioid_ingredients AS (
  SELECT concept_id
  FROM `{CDR}.concept`
  WHERE concept_class_id = 'Ingredient'
    AND vocabulary_id = 'RxNorm'
    AND LOWER(concept_name) IN (
      'morphine', 'oxycodone', 'hydromorphone', 'fentanyl', 'codeine',
      'tramadol', 'oxymorphone', 'buprenorphine', 'meperidine', 'methadone'
    )
),
opioid_ingredient_descendants AS (
  SELECT DISTINCT ca.descendant_concept_id AS ingredient_concept_id
  FROM `{CDR}.concept_ancestor` ca
  JOIN opioid_ingredients oi ON ca.ancestor_concept_id = oi.concept_id
),
opioid_drugs AS (
  SELECT DISTINCT ds.drug_concept_id
  FROM `{CDR}.drug_strength` ds
  JOIN opioid_ingredient_descendants oid ON ds.ingredient_concept_id = oid.ingredient_concept_id
),
first_opioid_exposure AS (
  SELECT
    de.person_id,
    MIN(DATE(de.drug_exposure_start_date)) AS first_drug_date
  FROM `{CDR}.drug_exposure` de
  JOIN opioid_drugs od ON de.drug_concept_id = od.drug_concept_id
  JOIN cohort co ON de.person_id = co.person_id
  WHERE DATE(de.drug_exposure_start_date) >= DATE(co.index_date)
  GROUP BY de.person_id
),

-- 3. Overdose events (after surgery AND after first drug exposure)
overdose_events AS (
  SELECT
    co.person_id,
    co.condition_start_date,
    c.index_date,
    foe.first_drug_date
  FROM `{CDR}.condition_occurrence` co
  JOIN cohort c ON co.person_id = c.person_id
  JOIN overdose_concepts oc ON co.condition_concept_id = oc.concept_id
  LEFT JOIN first_opioid_exposure foe ON co.person_id = foe.person_id
  WHERE DATE(co.condition_start_date) >= DATE(c.index_date)
    -- CRITICAL: Overdose must occur AFTER first drug exposure (if patient has any drugs)
    AND (foe.first_drug_date IS NULL OR DATE(co.condition_start_date) >= foe.first_drug_date)
)

-- 4. Binary outcome: overdose within 90 days of surgery AND after first drug
SELECT
  c.person_id AS person_id,
  CAST(
    CASE
      WHEN MIN(oe.condition_start_date) IS NOT NULL
       AND DATE_DIFF(
             DATE(MIN(oe.condition_start_date)),
             DATE(c.index_date),
             DAY
           ) BETWEEN 0 AND 90
      THEN 1
      ELSE 0
    END
  AS INT64) AS overdose
FROM cohort c
LEFT JOIN overdose_events oe ON c.person_id = oe.person_id
GROUP BY c.person_id, c.index_date
"""

# Run query
overdose = bq(sql_overdose)

# Demographics + TBIMS Embeddings

In [47]:
# Part 1: Demographics
sql_person = f"""
SELECT
  p.person_id,
  p.birth_datetime,
  g.concept_name AS gender,
  r.concept_name AS race,
  e.concept_name AS ethnicity,
  s.concept_name AS sex_at_birth
FROM `{CDR}.person` p
LEFT JOIN `{CDR}.concept` g ON p.gender_concept_id=g.concept_id
LEFT JOIN `{CDR}.concept` r ON p.race_concept_id=r.concept_id
LEFT JOIN `{CDR}.concept` e ON p.ethnicity_concept_id=e.concept_id
LEFT JOIN `{CDR}.concept` s ON p.sex_at_birth_concept_id=s.concept_id
WHERE p.person_id IN (SELECT person_id FROM ({sql_index_surgery}))
"""

person = bq(sql_person)

# Clean + derive demographics
demo = surgery_index_df.merge(person, on="person_id", how="left").copy()
demo["date_of_birth"] = pd.to_datetime(demo["birth_datetime"], errors="coerce", utc=True).dt.tz_localize(None)
demo["index_date_dt"] = pd.to_datetime(demo["index_date"], errors="coerce")
demo["age_at_index"] = ((demo["index_date_dt"] - demo["date_of_birth"]).dt.days / 365.25).round(1)

def norm(s): 
    return (s.astype(str).str.strip().str.lower()
            .replace({"nan":"unknown","none":"unknown"}))

demo["sex_cat"]  = norm(demo.get("sex_at_birth", pd.Series(["unknown"]*len(demo))))
demo["race_cat"] = norm(demo.get("race",        pd.Series(["unknown"]*len(demo))))
demo["eth_cat"]  = norm(demo.get("ethnicity",   pd.Series(["unknown"]*len(demo))))

common_races = {
    "white","black or african american","asian",
    "american indian or alaska native",
    "native hawaiian or other pacific islander",
    "unknown","more than one race"
}
demo["race_cat"] = demo["race_cat"].where(demo["race_cat"].isin(common_races), "other")

cats = pd.get_dummies(demo[["sex_cat","race_cat","eth_cat"]],
                      prefix=["sex","race","eth"], dtype="int8")

X_demo = pd.concat([demo[["person_id","age_at_index"]], cats], axis=1)

# ============================================================================
# FIX: Handle duplicate columns before merging
# ============================================================================
# Check which columns already exist in tx
existing_cols = set(tx.columns)
new_cols = set(X_demo.columns) - {'person_id'}  # person_id is the merge key

# Find overlapping columns (excluding person_id)
overlapping = existing_cols & new_cols

if len(overlapping) > 0:
    print(f"⚠️ WARNING: Found {len(overlapping)} duplicate columns in tx:")
    print(f"  {sorted(overlapping)[:10]}...")  # Show first 10
    print(f"  Dropping duplicates from tx before merge...")
    
    # Option 1: Drop duplicate columns from tx (keep new ones from X_demo)
    tx = tx.drop(columns=list(overlapping))
    
    # Option 2: Alternatively, you could drop from X_demo instead:
    # X_demo = X_demo.drop(columns=list(overlapping))
    # But Option 1 is safer if you want fresh demographics

# Merge demographics into tx
tx = tx.merge(X_demo, on="person_id", how="left")

# Part 2: Load TBIMS Embeddings
# Check if embeddings file exists (adjust path as needed)
import os
emb_file = "df_aug.parquet"  # Or your embeddings file path

if os.path.exists(emb_file):
    df_aug = pd.read_parquet(emb_file)
    print(f"Loaded embeddings from: {emb_file}")
    
    # Get embedding columns (emb_000 to emb_767)
    emb_cols = [c for c in df_aug.columns if c.startswith("emb_")]
    
    # Check for duplicate embedding columns too
    existing_emb = set(tx.columns) & set(emb_cols)
    if len(existing_emb) > 0:
        print(f"⚠️ WARNING: Found {len(existing_emb)} duplicate embedding columns, dropping from tx...")
        tx = tx.drop(columns=list(existing_emb))
    
    # Merge embeddings
    tx = tx.merge(df_aug[["person_id"] + emb_cols], on="person_id", how="left")
    
    print(f"  - Demographics: {len([c for c in X_demo.columns if c != 'person_id'])}")
    print(f"  - Embeddings: {len(emb_cols)}")
    print(f"  - Total features: {len([c for c in X_demo.columns if c != 'person_id']) + len(emb_cols)}")
else:
    print(f"⚠️ WARNING: Embeddings file not found: {emb_file}")
    print("  Continuing with demographics only")

print("\ntx FINAL shape:", tx.shape)
print(f"Total columns: {len(tx.columns)}")

⚠️ WARNING: Found 14 duplicate columns in tx:
  ['age_at_index', 'eth_hispanic or latino', 'eth_not hispanic or latino', 'eth_pmi: prefer not to answer', 'eth_pmi: skip', 'eth_what race ethnicity: race ethnicity none of these', 'race_asian', 'race_black or african american', 'race_other', 'race_white']...
  Dropping duplicates from tx before merge...
Loaded embeddings from: df_aug.parquet
⚠️ WARNING: Found 768 duplicate embedding columns, dropping from tx...
  - Demographics: 14
  - Embeddings: 768
  - Total features: 782

tx FINAL shape: (17101, 2570)
Total columns: 2570


# Average Treatment Effect (ATE) estimation

In [48]:
# ============================================================================
# STEP 1: SCREEN ALL DRUGS TO FIND CANDIDATES
# ============================================================================

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from scipy.stats import fisher_exact
import warnings
warnings.filterwarnings('ignore')

print("=" * 100)
print("STEP 1: SCREENING ALL DRUGS FOR TREATMENT EFFECTS")
print("=" * 100)

# Get all drug columns
all_drug_cols = [c for c in tx.columns if c.startswith('drug_')]
print(f"Screening {len(all_drug_cols)} drugs...\n")

screening_results = []

for col in all_drug_cols:
    t = tx[col].astype(int).values
    y = tx['overdose'].astype(int).values
    
    if t.sum() == 0 or t.sum() == len(t):
        continue
    
    n_treated = t.sum()
    n_control = len(t) - t.sum()
    
    treated_overdose = y[t==1].sum()
    treated_no_overdose = n_treated - treated_overdose
    control_overdose = y[t==0].sum()
    control_no_overdose = n_control - control_overdose
    
    if treated_overdose == 0 and control_overdose == 0:
        continue
    
    p_treated = treated_overdose / n_treated if n_treated > 0 else 0
    p_control = control_overdose / n_control if n_control > 0 else 0
    risk_diff = p_treated - p_control
    rr = p_treated / p_control if p_control > 0 else np.inf
    
    contingency = [[treated_overdose, treated_no_overdose],
                   [control_overdose, control_no_overdose]]
    
    try:
        oddsratio, p_value = fisher_exact(contingency, alternative='two-sided')
    except:
        oddsratio, p_value = np.nan, np.nan
    
    drug_id = int(col.replace('drug_', ''))
    drug_name = all_drug_names.get(drug_id, 'Unknown') if 'all_drug_names' in locals() else 'Unknown'
    
    screening_results.append({
        'drug_id': drug_id,
        'drug_name': drug_name,
        'col': col,
        'n_treated': n_treated,
        'n_control': n_control,
        'treated_overdose': treated_overdose,
        'control_overdose': control_overdose,
        'p_treated': p_treated,
        'p_control': p_control,
        'risk_diff': risk_diff,
        'relative_risk': rr,
        'odds_ratio': oddsratio,
        'p_value': p_value,
        'n_events_total': treated_overdose + control_overdose
    })

df_screen = pd.DataFrame(screening_results)

# Filter: Minimum sample size and events
min_treated = 100
min_control = 100
min_events = 5

df_screen_filtered = df_screen[
    (df_screen['n_treated'] >= min_treated) &
    (df_screen['n_control'] >= min_control) &
    (df_screen['n_events_total'] >= min_events)
].copy()

print(f"Found {len(df_screen_filtered)} drugs meeting criteria (≥{min_treated} treated, ≥{min_control} control, ≥{min_events} events)\n")

# Select ALL drugs that meet criteria (not just top 8)
selected_drugs = df_screen_filtered['col'].tolist()

print("=" * 100)
print(f"SELECTED {len(selected_drugs)} DRUGS FOR DETAILED ANALYSIS")
print("=" * 100)
for i, col in enumerate(selected_drugs, 1):
    row = df_screen_filtered[df_screen_filtered['col'] == col].iloc[0]
    print(f"{i}. {col} ({row['drug_name'][:60]})")
    print(f"   Risk diff: {row['risk_diff']:.4f}, RR: {row['relative_risk']:.2f}, p: {row['p_value']:.4f}")

# ============================================================================
# STEP 2: CREATE 1:1 BALANCED SAMPLES FOR ALL SELECTED DRUGS
# ============================================================================

print("\n" + "=" * 100)
print("STEP 2: CREATING 1:1 BALANCED SAMPLES FOR ALL SELECTED DRUGS")
print("=" * 100)

np.random.seed(42)
balanced_samples = {}

for col in selected_drugs:
    drug_id = int(col.replace('drug_', ''))
    drug_name = all_drug_names.get(drug_id, 'Unknown') if 'all_drug_names' in locals() else 'Unknown'
    
    treated_idx = tx[tx[col] == 1].index
    control_idx = tx[tx[col] == 0].index
    
    n_treated = len(treated_idx)
    n_control_available = len(control_idx)
    
    if n_control_available >= n_treated:
        sampled_control_idx = np.random.choice(control_idx, size=n_treated, replace=False)
        balanced_idx = np.concatenate([treated_idx, sampled_control_idx])
        n_control_used = n_treated
    else:
        balanced_idx = np.concatenate([treated_idx, control_idx])
        n_control_used = n_control_available
    
    balanced_samples[col] = {
        'drug_id': drug_id,
        'drug_name': drug_name,
        'balanced_indices': balanced_idx,
        'n_treated': n_treated,
        'n_control_used': n_control_used,
        'n_total': n_treated + n_control_used
    }
    
    print(f"{col:<25} {drug_name[:50]:<50} | Treated: {n_treated:5d} | Control: {n_control_used:5d} | Total: {n_treated + n_control_used:5d}")

# ============================================================================
# STEP 3: RUN ATE ANALYSIS ON ALL SELECTED DRUGS
# ============================================================================

print("\n" + "=" * 100)
print("STEP 3: RUNNING ATE ANALYSIS ON ALL SELECTED DRUGS")
print("=" * 100)

T_LIST = selected_drugs
Y_LIST = ['overdose']

results = []

for T in T_LIST:
    for Y in Y_LIST:
        print("\n" + "=" * 80)
        print(f"Running IPTW/DR for treatment = {T}, outcome = {Y}")
        print("=" * 80)
        
        if T not in balanced_samples:
            print(f"[WARN] No balanced sample found for {T}. Skipping.")
            results.append({
                "Treatment": T,
                "Outcome": Y,
                "N": 0,
                "IPTW_ATE": np.nan,
                "IPTW_CI_Lower": np.nan,
                "IPTW_CI_Upper": np.nan,
                "DR_ATE": np.nan
            })
            continue
        
        # Get balanced indices
        balanced_idx = balanced_samples[T]['balanced_indices']
        df_balanced = tx.loc[balanced_idx].copy()
        
        print(f"Using 1:1 balanced sample: {balanced_samples[T]['n_treated']} treated + {balanced_samples[T]['n_control_used']} control = {len(df_balanced)} total")

        # Covariates
        drop_cols = ['person_id', 'index_date', T, Y]
        drop_cols = [c for c in drop_cols if c in df_balanced.columns]
        
        Xcols = [c for c in df_balanced.columns if c not in drop_cols and df_balanced[c].dtype != 'O']
        X = df_balanced[Xcols].copy().fillna(0.0)
        t = df_balanced[T].astype(int).values
        y = df_balanced[Y].astype(int).values
        
        if t.sum() == 0 or t.sum() == len(t):
            print(f"[WARN] Skipping {T} — no variation")
            continue

        print(f"Treatment: {t.sum()} treated, {len(t) - t.sum()} control")
        print(f"Outcome: {y.sum()} positive, {len(y) - y.sum()} negative")

        # Propensity scores
        ps_model = LogisticRegression(max_iter=500, solver='lbfgs', random_state=42)
        ps_model.fit(X, t)
        ps = ps_model.predict_proba(X)[:,1].clip(1e-6, 1-1e-6)
        ps_auc = roc_auc_score(t, ps)
        print(f"PS AUC: {ps_auc:.3f}")

        # Overlap trimming
        keep = (ps >= 0.05) & (ps <= 0.95)
        X, t, y, ps = X[keep], t[keep], y[keep], ps[keep]
        n_total = len(y)
        n_treat, n_ctrl = t.sum(), (len(t) - t.sum())
        print(f"After trimming: {n_total} (treated={n_treat}, control={n_ctrl})")

        if n_treat == 0 or n_ctrl == 0:
            print(f"[WARN] Skipping — no units left after trimming")
            continue

        # IPTW weights
        w = np.where(t==1, 1/ps, 1/(1-ps))

        # IPTW ATE
        att_y1 = np.average(y[t==1], weights=w[t==1])
        att_y0 = np.average(y[t==0], weights=w[t==0])
        ate_iptw = att_y1 - att_y0
        print(f"IPTW ATE: {ate_iptw:.4f} (treated {att_y1:.4f} vs control {att_y0:.4f})")

        # Bootstrap CI
        rng = np.random.default_rng(42)
        idx = np.arange(len(y))
        ates = []
        for _ in range(1000):
            b = rng.choice(idx, size=len(idx), replace=True)
            tb, yb, wb = t[b], y[b], w[b]
            if tb.sum()==0 or (len(tb)-tb.sum())==0:
                continue
            att1 = np.average(yb[tb==1], weights=wb[tb==1])
            att0 = np.average(yb[tb==0], weights=wb[tb==0])
            ates.append(att1-att0)
        if len(ates) > 0:
            lo, hi = np.percentile(ates, [2.5, 97.5])
        else:
            lo, hi = np.nan, np.nan
        print(f"Bootstrap 95% CI: [{lo:.4f}, {hi:.4f}]")

        # AIPW/DR estimator
        if len(np.unique(y[t==1])) < 2 or len(np.unique(y[t==0])) < 2:
            print(f"[WARN] Skipping DR — insufficient outcome variation")
            ate_dr = np.nan
        else:
            mu1 = LogisticRegression(max_iter=1000, random_state=42).fit(X[t==1], y[t==1]).predict_proba(X)[:,1]
            mu0 = LogisticRegression(max_iter=1000, random_state=42).fit(X[t==0], y[t==0]).predict_proba(X)[:,1]
            dr = ( (t*(y-mu1))/ps + mu1 ) - ( ((1-t)*(y-mu0))/(1-ps) + mu0 )
            ate_dr = dr.mean()
            print(f"AIPW/DR ATE: {ate_dr:.4f}")

        results.append({
            "Treatment": T,
            "Drug_Name": balanced_samples[T]['drug_name'],
            "Outcome": Y,
            "N": n_total,
            "IPTW_ATE": ate_iptw,
            "IPTW_CI_Lower": lo,
            "IPTW_CI_Upper": hi,
            "DR_ATE": ate_dr,
            "Significant": (lo > 0) or (hi < 0)
        })

# Final results
results_df = pd.DataFrame(results)
print("\n" + "=" * 100)
print("FINAL RESULTS: ATE ANALYSIS ON ALL SELECTED DRUGS")
print("=" * 100)
print(results_df.to_string(index=False))

# Summary
print("\n" + "=" * 100)
print("SUMMARY")
print("=" * 100)
n_sig = results_df['Significant'].sum()
print(f"Total drugs analyzed: {len(results_df)}")
print(f"Significant effects: {n_sig}/{len(results_df)}")
if n_sig > 0:
    print("\nSignificant drugs:")
    for _, row in results_df[results_df['Significant']].iterrows():
        print(f"  {row['Treatment']} ({row['Drug_Name'][:50]}): ATE = {row['IPTW_ATE']:.4f}, CI = [{row['IPTW_CI_Lower']:.4f}, {row['IPTW_CI_Upper']:.4f}]")

STEP 1: SCREENING ALL DRUGS FOR TREATMENT EFFECTS
Screening 221 drugs...

Found 71 drugs meeting criteria (≥100 treated, ≥100 control, ≥5 events)

SELECTED 71 DRUGS FOR DETAILED ANALYSIS
1. drug_790240 (1 ML fentanyl 0.05 MG/ML Injection)
   Risk diff: -0.0043, RR: 0.00, p: 0.1208
2. drug_1102527 (meperidine)
   Risk diff: -0.0042, RR: 0.00, p: 0.4072
3. drug_1103314 (tramadol)
   Risk diff: 0.0024, RR: 1.60, p: 0.3602
4. drug_1103359 (Unknown)
   Risk diff: 0.0057, RR: 2.42, p: 0.3427
5. drug_1110410 (morphine)
   Risk diff: 0.0068, RR: 2.80, p: 0.0079
6. drug_1124957 (oxycodone)
   Risk diff: 0.0011, RR: 1.28, p: 0.4524
7. drug_1126658 (hydromorphone)
   Risk diff: 0.0002, RR: 1.05, p: 0.8550
8. drug_1154029 (fentanyl)
   Risk diff: -0.0015, RR: 0.65, p: 0.3703
9. drug_1154186 (fentanyl 0.1 MG)
   Risk diff: 0.0120, RR: 4.07, p: 0.0195
10. drug_1154321 (2 ML fentanyl 0.05 MG/ML Cartridge)
   Risk diff: 0.0029, RR: 1.72, p: 0.2622
11. drug_1201620 (codeine)
   Risk diff: 0.0058, RR: 2

# GATE_INDIVIDUAL

In [49]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import roc_auc_score
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Get significant drugs from ATE results
significant_drugs = results_df[results_df['Significant'] == True]['Treatment'].tolist()

print("=" * 100)
print("GATE ANALYSIS FOR SIGNIFICANT DRUGS")
print("=" * 100)
print(f"Analyzing {len(significant_drugs)} significant drugs\n")

# X-learner CATE function (same as TBI - 0.5/0.5 weighting)
def x_learner_cate_standard(X, t, y, ps):
    """
    X-learner CATE estimation with equal weighting (0.5/0.5)
    Same logic as TBI work
    """
    base_learner = HistGradientBoostingRegressor(
        max_iter=100,
        random_state=42,
        validation_fraction=0.1,
        n_iter_no_change=5
    )
    
    # Helper function
    def fit_or_constant(model, X_fit, y_fit):
        if len(np.unique(y_fit)) < 2:
            from sklearn.dummy import DummyRegressor
            return DummyRegressor(strategy='mean').fit(X_fit, y_fit)
        try:
            return model.fit(X_fit, y_fit)
        except:
            from sklearn.dummy import DummyRegressor
            return DummyRegressor(strategy='mean').fit(X_fit, y_fit)
    
    # Step 1: Fit outcome models
    m1 = fit_or_constant(
        base_learner.__class__(**base_learner.get_params()), 
        X[t==1], y[t==1]
    )
    m0 = fit_or_constant(
        base_learner.__class__(**base_learner.get_params()), 
        X[t==0], y[t==0]
    )
    
    # Step 2: Predict potential outcomes
    mu1_pred = np.clip(m1.predict(X), 0, 1)
    mu0_pred = np.clip(m0.predict(X), 0, 1)
    
    # Step 3: Construct pseudo-outcomes
    tau1 = y[t==1] - mu0_pred[t==1]  # Treated: Y - m₀(X)
    tau0 = mu1_pred[t==0] - y[t==0]  # Control: m₁(X) - Y
    
    # Step 4: Fit second-stage models
    h1 = fit_or_constant(
        base_learner.__class__(**base_learner.get_params()), 
        X[t==1], tau1
    )
    h0 = fit_or_constant(
        base_learner.__class__(**base_learner.get_params()), 
        X[t==0], tau0
    )
    
    # Step 5: Combine using equal weighting (0.5/0.5) - SAME AS TBI
    cate = 0.5 * h1.predict(X) + 0.5 * h0.predict(X)
    
    return cate, mu0_pred

def compute_gate(cate, mu0, n_quantiles=4):
    """Compute GATEs by baseline risk quartiles - with better error handling"""
    try:
        # Check if we have enough variation
        unique_vals = len(np.unique(mu0))
        
        if unique_vals < 2:
            print(f"      WARNING: Not enough variation in baseline risk (only {unique_vals} unique values)")
            return pd.DataFrame()
        
        # Try using fixed percentile thresholds (more robust)
        percentiles = np.percentile(mu0, [25, 50, 75])
        gate_rows = []
        
        # Q1: < 25th percentile
        mask1 = mu0 < percentiles[0]
        if mask1.sum() > 0:
            gate_rows.append({
                'Quartile': 'Q1_Low',
                'GATE': cate[mask1].mean(),
                'n': mask1.sum(),
                'mean_baseline_risk': mu0[mask1].mean()
            })
        
        # Q2: 25th to 50th percentile
        mask2 = (mu0 >= percentiles[0]) & (mu0 < percentiles[1])
        if mask2.sum() > 0:
            gate_rows.append({
                'Quartile': 'Q2_Med-Low',
                'GATE': cate[mask2].mean(),
                'n': mask2.sum(),
                'mean_baseline_risk': mu0[mask2].mean()
            })
        
        # Q3: 50th to 75th percentile
        mask3 = (mu0 >= percentiles[1]) & (mu0 < percentiles[2])
        if mask3.sum() > 0:
            gate_rows.append({
                'Quartile': 'Q3_Med-High',
                'GATE': cate[mask3].mean(),
                'n': mask3.sum(),
                'mean_baseline_risk': mu0[mask3].mean()
            })
        
        # Q4: >= 75th percentile
        mask4 = mu0 >= percentiles[2]
        if mask4.sum() > 0:
            gate_rows.append({
                'Quartile': 'Q4_High',
                'GATE': cate[mask4].mean(),
                'n': mask4.sum(),
                'mean_baseline_risk': mu0[mask4].mean()
            })
        
        if len(gate_rows) > 0:
            return pd.DataFrame(gate_rows)
        else:
            return pd.DataFrame()
            
    except Exception as e:
        print(f"      WARNING: Could not bin by risk: {e}")
        return pd.DataFrame()

def bootstrap_gate_ci(cate, mu0, n_quantiles=4, n_reps=1000, random_state=42):
    """Bootstrap confidence intervals for GATEs - with better error handling"""
    rng = np.random.default_rng(random_state)
    n = len(cate)
    
    gate_boot = []
    quartile_map = {"Q1_Low": 0, "Q2_Med-Low": 1, "Q3_Med-High": 2, "Q4_High": 3}
    
    for _ in tqdm(range(n_reps), desc="      Bootstrap CIs", leave=False):
        idx = rng.integers(0, n, n)
        cate_boot = cate[idx]
        mu0_boot = mu0[idx]
        
        try:
            # Use same percentile-based approach as compute_gate
            percentiles = np.percentile(mu0_boot, [25, 50, 75])
            
            # Q1
            mask1 = mu0_boot < percentiles[0]
            if mask1.sum() > 0:
                gate_boot.append({
                    'quartile': 0,
                    'gate': cate_boot[mask1].mean()
                })
            
            # Q2
            mask2 = (mu0_boot >= percentiles[0]) & (mu0_boot < percentiles[1])
            if mask2.sum() > 0:
                gate_boot.append({
                    'quartile': 1,
                    'gate': cate_boot[mask2].mean()
                })
            
            # Q3
            mask3 = (mu0_boot >= percentiles[1]) & (mu0_boot < percentiles[2])
            if mask3.sum() > 0:
                gate_boot.append({
                    'quartile': 2,
                    'gate': cate_boot[mask3].mean()
                })
            
            # Q4
            mask4 = mu0_boot >= percentiles[2]
            if mask4.sum() > 0:
                gate_boot.append({
                    'quartile': 3,
                    'gate': cate_boot[mask4].mean()
                })
        except:
            continue
    
    if len(gate_boot) == 0:
        return {}
    
    gate_boot_df = pd.DataFrame(gate_boot)
    
    # Compute CIs per quartile
    ci_dict = {}
    for q in range(n_quantiles):
        q_gates = gate_boot_df[gate_boot_df['quartile'] == q]['gate'].values
        if len(q_gates) > 0:
            ci_dict[q] = {
                'low': np.percentile(q_gates, 2.5),
                'high': np.percentile(q_gates, 97.5)
            }
    
    return ci_dict

# Run GATE analysis for each significant drug
gate_results = []

for T in significant_drugs:
    print("\n" + "=" * 100)
    print(f"GATE Analysis: {T} ({balanced_samples[T]['drug_name'][:60]})")
    print("=" * 100)
    
    if T not in balanced_samples:
        print(f"[WARN] No balanced sample found for {T}. Skipping.")
        continue
    
    # Get balanced indices
    balanced_idx = balanced_samples[T]['balanced_indices']
    df_balanced = tx.loc[balanced_idx].copy()
    
    print(f"Using 1:1 balanced sample: {balanced_samples[T]['n_treated']} treated + {balanced_samples[T]['n_control_used']} control = {len(df_balanced)} total")
    
    # Prepare covariates
    drop_cols = ['person_id', 'index_date', T, 'overdose']
    drop_cols = [c for c in drop_cols if c in df_balanced.columns]
    
    feature_cols = [c for c in df_balanced.columns 
                    if c not in drop_cols 
                    and df_balanced[c].dtype != 'O']
    
    X = df_balanced[feature_cols].fillna(0.0).astype("float32").values
    t = df_balanced[T].astype(int).values
    y = df_balanced['overdose'].astype(int).values
    
    print(f"\n[1] Data prepared: {len(feature_cols)} features, {len(X)} patients")
    print(f"   Treatment: {t.sum()} treated, {len(t) - t.sum()} control")
    print(f"   Outcome: {y.sum()} positive, {len(y) - y.sum()} negative")
    
    # Propensity score estimation
    print(f"\n[2] Estimating propensity scores...")
    ps_model = LogisticRegression(max_iter=1000, random_state=42)
    ps = ps_model.fit(X, t).predict_proba(X)[:, 1]
    ps = np.clip(ps, 1e-6, 1 - 1e-6)
    
    ps_auc = roc_auc_score(t, ps)
    print(f"   PS AUC: {ps_auc:.3f}")
    
    # Overlap trimming
    print(f"\n[3] Applying overlap trimming [0.05, 0.95]...")
    keep = (ps >= 0.05) & (ps <= 0.95)
    
    if keep.sum() == 0 or t[keep].sum() == 0 or t[keep].sum() == keep.sum():
        print("   WARNING: Insufficient overlap after trimming")
        continue
    
    X_overlap = X[keep]
    t_overlap = t[keep]
    y_overlap = y[keep]
    ps_overlap = ps[keep]
    
    print(f"   Kept: {keep.sum()}/{len(t)} ({keep.mean():.1%})")
    print(f"   Treated: {t_overlap.sum()}, Control: {(t_overlap==0).sum()}")
    
    # X-learner CATE
    print(f"\n[4] Running X-learner CATE estimation (0.5/0.5 weighting)...")
    cate, mu0 = x_learner_cate_standard(X_overlap, t_overlap, y_overlap, ps_overlap)
    
    # Compute overall ATE
    ate_overall = cate.mean()
    print(f"   Overall ATE: {ate_overall:.4f}")
    
    # Compute GATEs
    print(f"\n[5] Computing GATEs by baseline risk quartiles...")
    gate_df = compute_gate(cate, mu0, n_quantiles=4)
    
    if gate_df.empty:
        print("   WARNING: Could not compute GATEs - skipping this drug")
        continue
    
    # Bootstrap CIs
    print(f"\n[6] Computing bootstrap confidence intervals (1000 reps)...")
    ci_dict = bootstrap_gate_ci(cate, mu0, n_quantiles=4, n_reps=1000, random_state=42)
    
    # Merge CIs
    quartile_map = {"Q1_Low": 0, "Q2_Med-Low": 1, "Q3_Med-High": 2, "Q4_High": 3}
    for idx, row in gate_df.iterrows():
        q_label = row['Quartile']
        q_num = quartile_map.get(q_label, -1)
        if q_num >= 0 and q_num in ci_dict:
            gate_df.loc[idx, 'CI_low'] = ci_dict[q_num]['low']
            gate_df.loc[idx, 'CI_high'] = ci_dict[q_num]['high']
        else:
            gate_df.loc[idx, 'CI_low'] = np.nan
            gate_df.loc[idx, 'CI_high'] = np.nan
    
    gate_df['Treatment'] = T
    gate_df['Drug_Name'] = balanced_samples[T]['drug_name']
    gate_df['Outcome'] = 'overdose'
    gate_df['N'] = len(X_overlap)
    gate_df['ATE_Overall'] = ate_overall
    
    # Reorder columns
    col_order = ['Treatment', 'Drug_Name', 'Outcome', 'N', 'ATE_Overall', 'Quartile', 'n', 
                 'mean_baseline_risk', 'GATE', 'CI_low', 'CI_high']
    gate_df = gate_df[[c for c in col_order if c in gate_df.columns]]
    
    print("\n" + "=" * 80)
    print("GATE Results")
    print("=" * 80)
    print(gate_df.to_string(index=False))
    
    gate_results.append(gate_df)

# Combine all GATE results
if len(gate_results) > 0:
    all_gate_results = pd.concat(gate_results, ignore_index=True)
    
    print("\n" + "=" * 100)
    print("SUMMARY: ALL GATE RESULTS")
    print("=" * 100)
    print(all_gate_results.to_string(index=False))
    
    # Save results
    all_gate_results.to_csv('gate_results_opioid.csv', index=False)
    print(f"\nResults saved to 'gate_results_opioid.csv'")
else:
    print("\nNo GATE results generated.")

GATE ANALYSIS FOR SIGNIFICANT DRUGS
Analyzing 4 significant drugs


GATE Analysis: drug_1110410 (morphine)
Using 1:1 balanced sample: 855 treated + 855 control = 1710 total

[1] Data prepared: 2566 features, 1710 patients
   Treatment: 855 treated, 855 control
   Outcome: 10 positive, 1700 negative

[2] Estimating propensity scores...
   PS AUC: 0.919

[3] Applying overlap trimming [0.05, 0.95]...
   Kept: 1464/1710 (85.6%)
   Treated: 682, Control: 782

[4] Running X-learner CATE estimation (0.5/0.5 weighting)...
   Overall ATE: 0.0101

[5] Computing GATEs by baseline risk quartiles...

[6] Computing bootstrap confidence intervals (1000 reps)...



GATE Results
   Treatment Drug_Name  Outcome    N  ATE_Overall    Quartile   n  mean_baseline_risk     GATE   CI_low  CI_high
drug_1110410  morphine overdose 1464     0.010096  Q2_Med-Low 732        2.068865e-10 0.011739 0.009699 0.013761
drug_1110410  morphine overdose 1464     0.010096 Q3_Med-High 366        5.117200e-06 0.009003 0.006823 0.012369
drug_1110410  morphine overdose 1464     0.010096     Q4_High 366        5.556838e-03 0.007903 0.004385 0.011674

GATE Analysis: drug_19134047 (tramadol hydrochloride 50 MG Oral Tablet)
Using 1:1 balanced sample: 1052 treated + 1052 control = 2104 total

[1] Data prepared: 2566 features, 2104 patients
   Treatment: 1052 treated, 1052 control
   Outcome: 13 positive, 2091 negative

[2] Estimating propensity scores...
   PS AUC: 0.876

[3] Applying overlap trimming [0.05, 0.95]...
   Kept: 1979/2104 (94.1%)
   Treated: 965, Control: 1014

[4] Running X-learner CATE estimation (0.5/0.5 weighting)...
   Overall ATE: 0.0080

[5] Computing GATEs


GATE Results
    Treatment                                Drug_Name  Outcome    N  ATE_Overall    Quartile    n  mean_baseline_risk     GATE    CI_low  CI_high
drug_19134047 tramadol hydrochloride 50 MG Oral Tablet overdose 1979     0.008038 Q3_Med-High 1484            0.000092 0.010575  0.006703 0.011880
drug_19134047 tramadol hydrochloride 50 MG Oral Tablet overdose 1979     0.008038     Q4_High  495            0.014695 0.000430 -0.002886 0.004037

GATE Analysis: drug_40169988 (morphine sulfate 30 MG Extended Release Oral Tablet)
Using 1:1 balanced sample: 337 treated + 337 control = 674 total

[1] Data prepared: 2566 features, 674 patients
   Treatment: 337 treated, 337 control
   Outcome: 12 positive, 662 negative

[2] Estimating propensity scores...
   PS AUC: 0.942

[3] Applying overlap trimming [0.05, 0.95]...
   Kept: 569/674 (84.4%)
   Treated: 260, Control: 309

[4] Running X-learner CATE estimation (0.5/0.5 weighting)...
   Overall ATE: 0.0339

[5] Computing GATEs by baseli


GATE Results
    Treatment                                           Drug_Name  Outcome   N  ATE_Overall    Quartile   n  mean_baseline_risk     GATE   CI_low  CI_high
drug_40169988 morphine sulfate 30 MG Extended Release Oral Tablet overdose 569     0.033879      Q1_Low 135        4.104112e-08 0.033429 0.022535 0.044489
drug_40169988 morphine sulfate 30 MG Extended Release Oral Tablet overdose 569     0.033879  Q2_Med-Low 148        8.556718e-08 0.049457 0.033336 0.075322
drug_40169988 morphine sulfate 30 MG Extended Release Oral Tablet overdose 569     0.033879 Q3_Med-High 141        8.809649e-08 0.029557 0.014684 0.052159
drug_40169988 morphine sulfate 30 MG Extended Release Oral Tablet overdose 569     0.033879     Q4_High 145        1.676609e-02 0.022599 0.011578 0.033446

GATE Analysis: drug_40223140 (1 ML morphine sulfate 2 MG/ML Prefilled Syringe)
Using 1:1 balanced sample: 1143 treated + 1143 control = 2286 total

[1] Data prepared: 2566 features, 2286 patients
   Treatment: 


GATE Results
    Treatment                                       Drug_Name  Outcome    N  ATE_Overall    Quartile    n  mean_baseline_risk     GATE    CI_low  CI_high
drug_40223140 1 ML morphine sulfate 2 MG/ML Prefilled Syringe overdose 2056     0.008273  Q2_Med-Low 1028            0.000000 0.008501  0.007198 0.010387
drug_40223140 1 ML morphine sulfate 2 MG/ML Prefilled Syringe overdose 2056     0.008273 Q3_Med-High  513            0.000137 0.012406  0.008483 0.015156
drug_40223140 1 ML morphine sulfate 2 MG/ML Prefilled Syringe overdose 2056     0.008273     Q4_High  515            0.019282 0.003699 -0.000932 0.008256

SUMMARY: ALL GATE RESULTS
    Treatment                                           Drug_Name  Outcome    N  ATE_Overall    Quartile    n  mean_baseline_risk     GATE    CI_low  CI_high
 drug_1110410                                            morphine overdose 1464     0.010096  Q2_Med-Low  732        2.068865e-10 0.011739  0.009699 0.013761
 drug_1110410              

# GATE_Pairwise

In [50]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import roc_auc_score
from tqdm import tqdm
from itertools import combinations
import warnings
warnings.filterwarnings('ignore')

# Get significant drugs
significant_drugs = results_df[results_df['Significant'] == True]['Treatment'].tolist()

print("=" * 100)
print("PAIRWISE GATE ANALYSIS - Drug Combinations")
print("=" * 100)
print(f"Analyzing pairs from {len(significant_drugs)} significant drugs")
print(f"Total pairs: {len(list(combinations(significant_drugs, 2)))}\n")

# ============================================================================
# OPTIONAL: Query detailed exposures for overlap checking
# ============================================================================
# Set to True if you want to enforce overlapping/co-prescription windows
ENFORCE_OVERLAP = False  # Set to True to enable overlap checking
OVERLAP_WINDOW_DAYS = 7  # Days within which exposures must occur to count as "combination"

if ENFORCE_OVERLAP:
    print("Querying detailed exposure dates for overlap checking...")
    sql_pairwise_exposures = f"""
    WITH cohort AS ({sql_index_surgery}),
    opioid_ingredients AS (
      SELECT concept_id
      FROM `{CDR}.concept`
      WHERE concept_class_id = 'Ingredient'
        AND vocabulary_id = 'RxNorm'
        AND LOWER(concept_name) IN (
          'morphine', 'oxycodone', 'hydromorphone', 'fentanyl',
          'codeine', 'tramadol', 'oxymorphone', 'buprenorphine',
          'meperidine', 'methadone'
        )
    ),
    opioid_ingredient_descendants AS (
      SELECT DISTINCT ca.descendant_concept_id AS ingredient_concept_id
      FROM `{CDR}.concept_ancestor` ca
      JOIN opioid_ingredients oi ON ca.ancestor_concept_id = oi.concept_id
    ),
    opioid_drugs AS (
      SELECT DISTINCT ds.drug_concept_id
      FROM `{CDR}.drug_strength` ds
      JOIN opioid_ingredient_descendants oid
        ON ds.ingredient_concept_id = oid.ingredient_concept_id
    )
    SELECT
      c.person_id,
      de.drug_concept_id,
      DATE(de.drug_exposure_start_date) AS exposure_start,
      COALESCE(DATE(de.drug_exposure_end_date), DATE(de.drug_exposure_start_date)) AS exposure_end,
      DATE_DIFF(DATE(de.drug_exposure_start_date), DATE(c.index_date), DAY) AS days_from_surgery
    FROM `{CDR}.drug_exposure` de
    JOIN cohort c ON de.person_id = c.person_id
    JOIN opioid_drugs od ON de.drug_concept_id = od.drug_concept_id
    WHERE DATE(de.drug_exposure_start_date) >= DATE(c.index_date)
      AND DATE_DIFF(DATE(de.drug_exposure_start_date), DATE(c.index_date), DAY) <= 90
    """
    pairwise_exposures = bq(sql_pairwise_exposures)
    print(f"  Loaded {len(pairwise_exposures)} exposure records")
else:
    pairwise_exposures = None
    print("Overlap checking disabled (using 'ever exposed' definition)")

# Reuse the X-learner and GATE functions from before
def x_learner_cate_standard(X, t, y, ps):
    """X-learner CATE estimation with equal weighting (0.5/0.5)"""
    base_learner = HistGradientBoostingRegressor(
        max_iter=100,
        random_state=42,
        validation_fraction=0.1,
        n_iter_no_change=5
    )
    
    def fit_or_constant(model, X_fit, y_fit):
        if len(np.unique(y_fit)) < 2:
            from sklearn.dummy import DummyRegressor
            return DummyRegressor(strategy='mean').fit(X_fit, y_fit)
        try:
            return model.fit(X_fit, y_fit)
        except:
            from sklearn.dummy import DummyRegressor
            return DummyRegressor(strategy='mean').fit(X_fit, y_fit)
    
    m1 = fit_or_constant(base_learner.__class__(**base_learner.get_params()), X[t==1], y[t==1])
    m0 = fit_or_constant(base_learner.__class__(**base_learner.get_params()), X[t==0], y[t==0])
    
    mu1_pred = np.clip(m1.predict(X), 0, 1)
    mu0_pred = np.clip(m0.predict(X), 0, 1)
    
    tau1 = y[t==1] - mu0_pred[t==1]
    tau0 = mu1_pred[t==0] - y[t==0]
    
    h1 = fit_or_constant(base_learner.__class__(**base_learner.get_params()), X[t==1], tau1)
    h0 = fit_or_constant(base_learner.__class__(**base_learner.get_params()), X[t==0], tau0)
    
    cate = 0.5 * h1.predict(X) + 0.5 * h0.predict(X)
    
    return cate, mu0_pred

def compute_gate(cate, mu0, n_quantiles=4):
    """Compute GATEs by baseline risk quartiles"""
    try:
        percentiles = np.percentile(mu0, [25, 50, 75])
        gate_rows = []
        
        mask1 = mu0 < percentiles[0]
        if mask1.sum() > 0:
            gate_rows.append({
                'Quartile': 'Q1_Low',
                'GATE': cate[mask1].mean(),
                'n': mask1.sum(),
                'mean_baseline_risk': mu0[mask1].mean()
            })
        
        mask2 = (mu0 >= percentiles[0]) & (mu0 < percentiles[1])
        if mask2.sum() > 0:
            gate_rows.append({
                'Quartile': 'Q2_Med-Low',
                'GATE': cate[mask2].mean(),
                'n': mask2.sum(),
                'mean_baseline_risk': mu0[mask2].mean()
            })
        
        mask3 = (mu0 >= percentiles[1]) & (mu0 < percentiles[2])
        if mask3.sum() > 0:
            gate_rows.append({
                'Quartile': 'Q3_Med-High',
                'GATE': cate[mask3].mean(),
                'n': mask3.sum(),
                'mean_baseline_risk': mu0[mask3].mean()
            })
        
        mask4 = mu0 >= percentiles[2]
        if mask4.sum() > 0:
            gate_rows.append({
                'Quartile': 'Q4_High',
                'GATE': cate[mask4].mean(),
                'n': mask4.sum(),
                'mean_baseline_risk': mu0[mask4].mean()
            })
        
        if len(gate_rows) > 0:
            return pd.DataFrame(gate_rows)
        else:
            return pd.DataFrame()
    except Exception as e:
        return pd.DataFrame()

def bootstrap_gate_ci(cate, mu0, n_quantiles=4, n_reps=1000, random_state=42):
    """Bootstrap confidence intervals for GATEs"""
    rng = np.random.default_rng(random_state)
    n = len(cate)
    
    gate_boot = []
    
    for _ in tqdm(range(n_reps), desc="      Bootstrap CIs", leave=False):
        idx = rng.integers(0, n, n)
        cate_boot = cate[idx]
        mu0_boot = mu0[idx]
        
        try:
            percentiles = np.percentile(mu0_boot, [25, 50, 75])
            
            mask1 = mu0_boot < percentiles[0]
            if mask1.sum() > 0:
                gate_boot.append({'quartile': 0, 'gate': cate_boot[mask1].mean()})
            
            mask2 = (mu0_boot >= percentiles[0]) & (mu0_boot < percentiles[1])
            if mask2.sum() > 0:
                gate_boot.append({'quartile': 1, 'gate': cate_boot[mask2].mean()})
            
            mask3 = (mu0_boot >= percentiles[1]) & (mu0_boot < percentiles[2])
            if mask3.sum() > 0:
                gate_boot.append({'quartile': 2, 'gate': cate_boot[mask3].mean()})
            
            mask4 = mu0_boot >= percentiles[2]
            if mask4.sum() > 0:
                gate_boot.append({'quartile': 3, 'gate': cate_boot[mask4].mean()})
        except:
            continue
    
    if len(gate_boot) == 0:
        return {}
    
    gate_boot_df = pd.DataFrame(gate_boot)
    
    ci_dict = {}
    for q in range(n_quantiles):
        q_gates = gate_boot_df[gate_boot_df['quartile'] == q]['gate'].values
        if len(q_gates) > 0:
            ci_dict[q] = {
                'low': np.percentile(q_gates, 2.5),
                'high': np.percentile(q_gates, 97.5)
            }
    
    return ci_dict

def check_overlap(person_id, drug1_id, drug2_id, exposures_df, window_days=7):
    """Check if person has overlapping exposures to drug1 and drug2 within window_days"""
    if exposures_df is None:
        return False
    
    d1_exposures = exposures_df[
        (exposures_df['person_id'] == person_id) & 
        (exposures_df['drug_concept_id'] == drug1_id)
    ]
    d2_exposures = exposures_df[
        (exposures_df['person_id'] == person_id) & 
        (exposures_df['drug_concept_id'] == drug2_id)
    ]
    
    if len(d1_exposures) == 0 or len(d2_exposures) == 0:
        return False
    
    for _, d1_row in d1_exposures.iterrows():
        d1_start = d1_row['days_from_surgery']
        d1_end = d1_start + (pd.to_datetime(d1_row['exposure_end']) - pd.to_datetime(d1_row['exposure_start'])).days
        
        for _, d2_row in d2_exposures.iterrows():
            d2_start = d2_row['days_from_surgery']
            d2_end = d2_start + (pd.to_datetime(d2_row['exposure_end']) - pd.to_datetime(d2_row['exposure_start'])).days
            
            # Check if ranges overlap or are within window
            if (d1_start <= d2_end + window_days) and (d2_start <= d1_end + window_days):
                return True
    
    return False

# Generate all pairs
drug_pairs = list(combinations(significant_drugs, 2))

# Limit to top pairs if too many (optional - remove this if you want all pairs)
MAX_PAIRS = 50  # Adjust as needed
if len(drug_pairs) > MAX_PAIRS:
    print(f"Limiting to top {MAX_PAIRS} pairs (out of {len(drug_pairs)} total)")
    drug_pairs = drug_pairs[:MAX_PAIRS]

print(f"\nAnalyzing {len(drug_pairs)} drug pairs...\n")

pairwise_results = []

for idx, (drug1, drug2) in enumerate(drug_pairs, 1):
    print("\n" + "=" * 100)
    print(f"Pair {idx}/{len(drug_pairs)}: {drug1} + {drug2}")
    print("=" * 100)
    
    drug1_name = balanced_samples[drug1]['drug_name'] if drug1 in balanced_samples else drug1
    drug2_name = balanced_samples[drug2]['drug_name'] if drug2 in balanced_samples else drug2
    print(f"  {drug1_name[:50]}")
    print(f"  {drug2_name[:50]}")
    
    # Extract drug IDs for overlap checking
    drug1_id = int(drug1.replace('drug_', ''))
    drug2_id = int(drug2.replace('drug_', ''))
    
    # ========================================================================
    # FIX #2: Strict "neither" control - exclude ALL opioids
    # ========================================================================
    # Get all drug columns (excluding person_id, index_date, overdose, demo/embedding cols)
    drug_cols = [c for c in tx.columns if c.startswith('drug_')]
    
    # "Both" = exposed to both drug1 AND drug2
    if ENFORCE_OVERLAP and pairwise_exposures is not None:
        # Check for overlapping exposures
        both_person_ids = []
        for person_id in tx['person_id'].unique():
            if (tx.loc[tx['person_id'] == person_id, drug1].iloc[0] == 1 and
                tx.loc[tx['person_id'] == person_id, drug2].iloc[0] == 1):
                if check_overlap(person_id, drug1_id, drug2_id, pairwise_exposures, OVERLAP_WINDOW_DAYS):
                    both_person_ids.append(person_id)
        both_drugs = tx['person_id'].isin(both_person_ids)
        print(f"  [Overlap check enabled] Both drugs with overlap: {both_drugs.sum()}")
    else:
        # Simple "ever exposed to both" definition
        both_drugs = (tx[drug1] == 1) & (tx[drug2] == 1)
    
    # "Neither" = NO exposure to ANY opioid drug (strict control)
    neither_drug = (tx[drug_cols].sum(axis=1) == 0)
    
    n_both = both_drugs.sum()
    n_neither = neither_drug.sum()
    
    print(f"\n  Patients with both drugs: {n_both}")
    print(f"  Patients with NO opioids (strict control): {n_neither}")
    
    # Need sufficient sample size
    min_samples = 50
    if n_both < min_samples or n_neither < min_samples:
        print(f"  [SKIP] Insufficient sample size (need ≥{min_samples} in each group)")
        continue
    
    # Create 1:1 balanced sample
    both_idx = tx[both_drugs].index
    neither_idx = tx[neither_drug].index
    
    # Sample to match smaller group
    n_sample = min(n_both, n_neither)
    
    np.random.seed(42)
    if n_both >= n_sample:
        sampled_both_idx = np.random.choice(both_idx, size=n_sample, replace=False)
    else:
        sampled_both_idx = both_idx
    
    if n_neither >= n_sample:
        sampled_neither_idx = np.random.choice(neither_idx, size=n_sample, replace=False)
    else:
        sampled_neither_idx = neither_idx
    
    balanced_idx = np.concatenate([sampled_both_idx, sampled_neither_idx])
    df_balanced = tx.loc[balanced_idx].copy()
    
    # Create treatment variable: 1 = both drugs, 0 = no opioids
    t_pair = ((df_balanced[drug1] == 1) & (df_balanced[drug2] == 1)).astype(int).values
    
    print(f"  Balanced sample: {t_pair.sum()} treated (both), {len(t_pair) - t_pair.sum()} control (no opioids)")
    
    # Prepare covariates
    drop_cols = ['person_id', 'index_date', drug1, drug2, 'overdose']
    drop_cols = [c for c in drop_cols if c in df_balanced.columns]
    
    feature_cols = [c for c in df_balanced.columns 
                    if c not in drop_cols 
                    and df_balanced[c].dtype != 'O']
    
    X = df_balanced[feature_cols].fillna(0.0).astype("float32").values
    y = df_balanced['overdose'].astype(int).values
    
    print(f"  Features: {len(feature_cols)}, Patients: {len(X)}")
    print(f"  Outcome: {y.sum()} positive, {len(y) - y.sum()} negative")
    
    # Check variation
    if t_pair.sum() == 0 or t_pair.sum() == len(t_pair):
        print(f"  [SKIP] No variation in treatment")
        continue
    
    # Propensity score
    print(f"\n  [1] Estimating propensity scores...")
    ps_model = LogisticRegression(max_iter=1000, random_state=42)
    ps = ps_model.fit(X, t_pair).predict_proba(X)[:, 1]
    ps = np.clip(ps, 1e-6, 1 - 1e-6)
    
    ps_auc = roc_auc_score(t_pair, ps)
    print(f"      PS AUC: {ps_auc:.3f}")
    
    # Overlap trimming
    print(f"  [2] Applying overlap trimming...")
    keep = (ps >= 0.05) & (ps <= 0.95)
    
    if keep.sum() == 0 or t_pair[keep].sum() == 0 or t_pair[keep].sum() == keep.sum():
        print(f"      [SKIP] Insufficient overlap after trimming")
        continue
    
    X_overlap = X[keep]
    t_overlap = t_pair[keep]
    y_overlap = y[keep]
    ps_overlap = ps[keep]
    
    print(f"      Kept: {keep.sum()}/{len(t_pair)} ({keep.mean():.1%})")
    print(f"      Treated: {t_overlap.sum()}, Control: {(t_overlap==0).sum()}")
    
    # X-learner CATE
    print(f"  [3] Running X-learner CATE estimation...")
    cate, mu0 = x_learner_cate_standard(X_overlap, t_overlap, y_overlap, ps_overlap)
    
    ate_overall = cate.mean()
    print(f"      Overall ATE: {ate_overall:.4f}")
    
    # Compute GATEs
    print(f"  [4] Computing GATEs...")
    gate_df = compute_gate(cate, mu0, n_quantiles=4)
    
    if gate_df.empty:
        print(f"      [SKIP] Could not compute GATEs")
        continue
    
    # Bootstrap CIs
    print(f"  [5] Computing bootstrap CIs...")
    ci_dict = bootstrap_gate_ci(cate, mu0, n_quantiles=4, n_reps=1000, random_state=42)
    
    # Merge CIs
    quartile_map = {"Q1_Low": 0, "Q2_Med-Low": 1, "Q3_Med-High": 2, "Q4_High": 3}
    for idx_row, row in gate_df.iterrows():
        q_label = row['Quartile']
        q_num = quartile_map.get(q_label, -1)
        if q_num >= 0 and q_num in ci_dict:
            gate_df.loc[idx_row, 'CI_low'] = ci_dict[q_num]['low']
            gate_df.loc[idx_row, 'CI_high'] = ci_dict[q_num]['high']
        else:
            gate_df.loc[idx_row, 'CI_low'] = np.nan
            gate_df.loc[idx_row, 'CI_high'] = np.nan
    
    gate_df['Drug1'] = drug1
    gate_df['Drug1_Name'] = drug1_name
    gate_df['Drug2'] = drug2
    gate_df['Drug2_Name'] = drug2_name
    gate_df['Treatment'] = f"{drug1} + {drug2}"
    gate_df['Outcome'] = 'overdose'
    gate_df['N'] = len(X_overlap)
    gate_df['ATE_Overall'] = ate_overall
    gate_df['Overlap_Enforced'] = ENFORCE_OVERLAP
    if ENFORCE_OVERLAP:
        gate_df['Overlap_Window_Days'] = OVERLAP_WINDOW_DAYS
    
    # Reorder columns
    col_order = ['Drug1', 'Drug1_Name', 'Drug2', 'Drug2_Name', 'Treatment', 'Outcome', 'N', 
                 'ATE_Overall', 'Quartile', 'n', 'mean_baseline_risk', 'GATE', 'CI_low', 'CI_high']
    if ENFORCE_OVERLAP:
        col_order.extend(['Overlap_Enforced', 'Overlap_Window_Days'])
    gate_df = gate_df[[c for c in col_order if c in gate_df.columns]]
    
    print(f"\n  GATE Results:")
    print(gate_df[['Quartile', 'n', 'GATE', 'CI_low', 'CI_high']].to_string(index=False))
    
    pairwise_results.append(gate_df)

# Combine all pairwise results
if len(pairwise_results) > 0:
    all_pairwise_results = pd.concat(pairwise_results, ignore_index=True)
    
    print("\n" + "=" * 100)
    print("SUMMARY: ALL PAIRWISE GATE RESULTS")
    print("=" * 100)
    print(all_pairwise_results.to_string(index=False))
    
    # Save results
    all_pairwise_results.to_csv('gate_results_pairwise_opioid.csv', index=False)
    print(f"\nResults saved to 'gate_results_pairwise_opioid.csv'")
    
    # Summary statistics
    print("\n" + "=" * 100)
    print("PAIRWISE ANALYSIS SUMMARY")
    print("=" * 100)
    print(f"Total pairs analyzed: {len(pairwise_results)}")
    print(f"Control group definition: NO opioids at all (strict)")
    if ENFORCE_OVERLAP:
        print(f"Overlap enforcement: Enabled (window = {OVERLAP_WINDOW_DAYS} days)")
    else:
        print(f"Overlap enforcement: Disabled (ever exposed definition)")
    
    print(f"\nPairs with significant effects (CI doesn't contain 0):")
    significant_count = 0
    for _, row in all_pairwise_results.iterrows():
        if not pd.isna(row['CI_low']) and not pd.isna(row['CI_high']):
            if (row['CI_low'] > 0) or (row['CI_high'] < 0):
                significant_count += 1
                print(f"  {row['Drug1_Name'][:30]} + {row['Drug2_Name'][:30]}: "
                      f"{row['Quartile']} GATE={row['GATE']:.4f} [{row['CI_low']:.4f}, {row['CI_high']:.4f}]")
    
    print(f"\nTotal significant effects: {significant_count}")
else:
    print("\nNo pairwise results generated.")

PAIRWISE GATE ANALYSIS - Drug Combinations
Analyzing pairs from 4 significant drugs
Total pairs: 6

Overlap checking disabled (using 'ever exposed' definition)

Analyzing 6 drug pairs...


Pair 1/6: drug_1110410 + drug_19134047
  morphine
  tramadol hydrochloride 50 MG Oral Tablet

  Patients with both drugs: 43
  Patients with NO opioids (strict control): 5728
  [SKIP] Insufficient sample size (need ≥50 in each group)

Pair 2/6: drug_1110410 + drug_40169988
  morphine
  morphine sulfate 30 MG Extended Release Oral Table

  Patients with both drugs: 63
  Patients with NO opioids (strict control): 5728
  Balanced sample: 63 treated (both), 63 control (no opioids)
  Features: 2565, Patients: 126
  Outcome: 3 positive, 123 negative

  [1] Estimating propensity scores...
      PS AUC: 1.000
  [2] Applying overlap trimming...
      Kept: 84/126 (66.7%)
      Treated: 29, Control: 55
  [3] Running X-learner CATE estimation...
      Overall ATE: 0.0000
  [4] Computing GATEs...
  [5] Computing


  GATE Results:
Quartile  n  GATE  CI_low  CI_high
 Q4_High 84   0.0     0.0      0.0

Pair 3/6: drug_1110410 + drug_40223140
  morphine
  1 ML morphine sulfate 2 MG/ML Prefilled Syringe

  Patients with both drugs: 166
  Patients with NO opioids (strict control): 5728
  Balanced sample: 166 treated (both), 166 control (no opioids)
  Features: 2565, Patients: 332
  Outcome: 1 positive, 331 negative

  [1] Estimating propensity scores...
      PS AUC: 0.997
  [2] Applying overlap trimming...
      Kept: 210/332 (63.3%)
      Treated: 55, Control: 155
  [3] Running X-learner CATE estimation...
      Overall ATE: 0.0000
  [4] Computing GATEs...
  [5] Computing bootstrap CIs...



  GATE Results:
Quartile   n  GATE  CI_low  CI_high
 Q4_High 210   0.0     0.0      0.0

Pair 4/6: drug_19134047 + drug_40169988
  tramadol hydrochloride 50 MG Oral Tablet
  morphine sulfate 30 MG Extended Release Oral Table

  Patients with both drugs: 21
  Patients with NO opioids (strict control): 5728
  [SKIP] Insufficient sample size (need ≥50 in each group)

Pair 5/6: drug_19134047 + drug_40223140
  tramadol hydrochloride 50 MG Oral Tablet
  1 ML morphine sulfate 2 MG/ML Prefilled Syringe

  Patients with both drugs: 155
  Patients with NO opioids (strict control): 5728
  Balanced sample: 155 treated (both), 155 control (no opioids)
  Features: 2565, Patients: 310
  Outcome: 3 positive, 307 negative

  [1] Estimating propensity scores...
      PS AUC: 0.997
  [2] Applying overlap trimming...
      Kept: 184/310 (59.4%)
      Treated: 66, Control: 118
  [3] Running X-learner CATE estimation...
      Overall ATE: 0.0219
  [4] Computing GATEs...
  [5] Computing bootstrap CIs...



  GATE Results:
Quartile   n     GATE   CI_low  CI_high
 Q4_High 184 0.021926 0.017618 0.026945

Pair 6/6: drug_40169988 + drug_40223140
  morphine sulfate 30 MG Extended Release Oral Table
  1 ML morphine sulfate 2 MG/ML Prefilled Syringe



  Patients with both drugs: 55
  Patients with NO opioids (strict control): 5728
  Balanced sample: 55 treated (both), 55 control (no opioids)
  Features: 2565, Patients: 110
  Outcome: 3 positive, 107 negative

  [1] Estimating propensity scores...
      PS AUC: 0.995
  [2] Applying overlap trimming...
      Kept: 82/110 (74.5%)
      Treated: 33, Control: 49
  [3] Running X-learner CATE estimation...
      Overall ATE: 0.0909
  [4] Computing GATEs...
  [5] Computing bootstrap CIs...



  GATE Results:
Quartile  n     GATE   CI_low  CI_high
 Q4_High 82 0.090909 0.090909 0.090909

SUMMARY: ALL PAIRWISE GATE RESULTS
        Drug1                                          Drug1_Name         Drug2                                          Drug2_Name                     Treatment  Outcome   N  ATE_Overall Quartile   n  mean_baseline_risk     GATE   CI_low  CI_high
 drug_1110410                                            morphine drug_40169988 morphine sulfate 30 MG Extended Release Oral Tablet  drug_1110410 + drug_40169988 overdose  84     0.000000  Q4_High  84                 0.0 0.000000 0.000000 0.000000
 drug_1110410                                            morphine drug_40223140     1 ML morphine sulfate 2 MG/ML Prefilled Syringe  drug_1110410 + drug_40223140 overdose 210     0.000000  Q4_High 210                 0.0 0.000000 0.000000 0.000000
drug_19134047            tramadol hydrochloride 50 MG Oral Tablet drug_40223140     1 ML morphine sulfate 2 MG/ML Prefilled S

# rTMLE-GATE_INDIVIDUAL

In [53]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import roc_auc_score
from scipy.optimize import minimize
from scipy.special import expit, logit
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')


def log_like(beta, Y, X, wt=None):
    if wt is None:
        wt = np.ones(len(Y))
    pi = expit(X @ beta)
    pi = np.clip(pi, np.finfo(float).eps, 1 - np.finfo(float).eps)
    log_like = np.sum(wt * (Y * np.log(pi) + (1 - Y) * np.log(1 - pi)))
    return -log_like

def grad(beta, Y, X, wt=None):
    if wt is None:
        wt = np.ones(len(Y))
    pi = expit(X @ beta)
    pi = np.clip(pi, np.finfo(float).eps, 1 - np.finfo(float).eps)
    resid = wt * (Y - pi)
    gr = X.T @ resid
    return -gr

def tmle_log_like(eps, Y, Q_AW, H_0W, H_1W, wt=None):
    if wt is None:
        wt = np.ones(len(Y))
    pi = expit(logit(Q_AW) + eps[0] * H_0W + eps[1] * H_1W)
    pi = np.clip(pi, np.finfo(float).eps, 1 - np.finfo(float).eps)
    log_like = np.sum(wt * (Y * np.log(pi) + (1 - Y) * np.log(1 - pi)))
    if np.isnan(log_like) or np.isinf(log_like):
        return 99
    return -log_like

def tmle_grad(eps, Y, Q_AW, H_0W, H_1W, wt=None):
    if wt is None:
        wt = np.ones(len(Y))
    pi = expit(logit(Q_AW) + eps[0] * H_0W + eps[1] * H_1W)
    pi = np.clip(pi, np.finfo(float).eps, 1 - np.finfo(float).eps)
    resid = wt * (Y - pi)
    gr = np.array([np.sum(H_0W * resid), np.sum(H_1W * resid)])
    if np.any(np.isnan(gr)) or np.any(np.isinf(gr)):
        return np.array([99, 99])
    return -gr

# ============================================================================
# OUTCOME MODELING FUNCTIONS: MLE, TMLE, rTMLE
# ============================================================================

def mle_outcome_model(W, A, Y, outcome_vars, weights=None):
    """MLE: Simple logistic regression for outcome modeling"""
    if weights is None:
        weights = np.ones(len(Y))
    
    data_temp = pd.DataFrame(W)
    data_temp['A'] = A
    
    X_model = data_temp[outcome_vars + ['A']].values
    X_model = np.column_stack([np.ones(len(X_model)), X_model])
    
    try:
        model = LogisticRegression(max_iter=1000, fit_intercept=False)
        model.fit(X_model, Y, sample_weight=weights)
        beta = model.coef_[0]
    except:
        try:
            result = minimize(
                log_like,
                x0=np.zeros(X_model.shape[1]),
                args=(Y, X_model, weights),
                method='BFGS',
                jac=grad,
                options={'maxiter': 500}
            )
            beta = result.x
        except:
            beta = np.zeros(X_model.shape[1])
    
    X1 = X_model.copy()
    X0 = X_model.copy()
    A_idx = len(outcome_vars) + 1
    X1[:, A_idx] = 1
    X0[:, A_idx] = 0
    
    Qbar1W = expit(X1 @ beta)
    Qbar0W = expit(X0 @ beta)
    
    return Qbar1W, Qbar0W

def tmle_outcome_model(W, A, Y, outcome_vars, propensity_vars, weights=None):
    """TMLE: Targeted Maximum Likelihood Estimation for outcome modeling"""
    if weights is None:
        weights = np.ones(len(Y))
    
    data_temp = pd.DataFrame(W)
    data_temp['A'] = A
    
    X_model = data_temp[outcome_vars + ['A']].values
    X_model = np.column_stack([np.ones(len(X_model)), X_model])
    
    try:
        result = minimize(
            log_like,
            x0=np.zeros(X_model.shape[1]),
            args=(Y, X_model, weights),
            method='BFGS',
            jac=grad,
            options={'maxiter': 500}
        )
        beta = result.x
    except:
        model = LogisticRegression(max_iter=1000, fit_intercept=False)
        model.fit(X_model, Y, sample_weight=weights)
        beta = model.coef_[0]
    
    X1 = X_model.copy()
    X0 = X_model.copy()
    A_idx = len(outcome_vars) + 1
    X1[:, A_idx] = 1
    X0[:, A_idx] = 0
    
    QbarAW = expit(X_model @ beta)
    Qbar1W = expit(X1 @ beta)
    Qbar0W = expit(X0 @ beta)
    
    # Propensity score
    X_g = W[propensity_vars].values
    g_model = LogisticRegression(fit_intercept=True, max_iter=1000)
    g_model.fit(X_g, A, sample_weight=weights)
    g1W = g_model.predict_proba(X_g)[:, 1]
    g0W = 1 - g1W
    
    H_1W = A / np.clip(g1W, 1e-6, 1-1e-6)
    H_0W = (1 - A) / np.clip(g0W, 1e-6, 1-1e-6)
    
    # Targeting step
    try:
        result_eps = minimize(
            tmle_log_like,
            x0=[0, 0],
            args=(Y, QbarAW, H_0W, H_1W, weights),
            method='BFGS',
            jac=tmle_grad,
            options={'maxiter': 500}
        )
        eps = result_eps.x
    except:
        eps = [0, 0]
    
    QbarAW_updated = expit(logit(QbarAW) + eps[0] * H_0W + eps[1] * H_1W)
    Qbar0W_updated = expit(logit(Qbar0W) + eps[0] / np.clip(g0W, 1e-6, 1-1e-6))
    Qbar1W_updated = expit(logit(Qbar1W) + eps[1] / np.clip(g1W, 1e-6, 1-1e-6))
    
    return Qbar1W_updated, Qbar0W_updated

def rtmle_outcome_model(W, A, Y, a, b, outcome_vars, propensity_vars, weights=None):
    """rTMLE: Rare outcomes Targeted Maximum Likelihood Estimation"""
    if weights is None:
        weights = np.ones(len(Y))
    
    Y_tilde = (Y - a) / (b - a)
    
    data_temp = pd.DataFrame(W)
    data_temp['A'] = A
    
    X_model = data_temp[outcome_vars + ['A']].values
    X_model = np.column_stack([np.ones(len(X_model)), X_model])
    
    try:
        result = minimize(
            log_like,
            x0=np.zeros(X_model.shape[1]),
            args=(Y_tilde, X_model, weights),
            method='BFGS',
            jac=grad,
            options={'maxiter': 500}
        )
        beta = result.x
    except:
        model = LogisticRegression(max_iter=1000, fit_intercept=False)
        model.fit(X_model, Y_tilde, sample_weight=weights)
        beta = model.coef_[0]
    
    X1 = X_model.copy()
    X0 = X_model.copy()
    A_idx = len(outcome_vars) + 1
    X1[:, A_idx] = 1
    X0[:, A_idx] = 0
    
    QbarAW = expit(X_model @ beta)
    Qbar1W_tilde = expit(X1 @ beta)
    Qbar0W_tilde = expit(X0 @ beta)
    
    X_g = W[propensity_vars].values
    g_model = LogisticRegression(fit_intercept=True, max_iter=1000)
    g_model.fit(X_g, A, sample_weight=weights)
    g1W = g_model.predict_proba(X_g)[:, 1]
    g0W = 1 - g1W
    
    H_1W = A / np.clip(g1W, 1e-6, 1-1e-6)
    H_0W = (1 - A) / np.clip(g0W, 1e-6, 1-1e-6)
    
    try:
        result_eps = minimize(
            tmle_log_like,
            x0=[0, 0],
            args=(Y_tilde, QbarAW, H_0W, H_1W, weights),
            method='BFGS',
            jac=tmle_grad,
            options={'maxiter': 500}
        )
        eps = result_eps.x
    except:
        eps = [0, 0]
    
    QbarAW_updated = expit(logit(QbarAW) + eps[0] * H_0W + eps[1] * H_1W)
    Qbar0W_updated = expit(logit(Qbar0W_tilde) + eps[0] / np.clip(g0W, 1e-6, 1-1e-6))
    Qbar1W_updated = expit(logit(Qbar1W_tilde) + eps[1] / np.clip(g1W, 1e-6, 1-1e-6))
    
    Qbar1W = Qbar1W_updated * (b - a) + a
    Qbar0W = Qbar0W_updated * (b - a) + a
    
    Qbar1W = np.clip(Qbar1W, a, b)
    Qbar0W = np.clip(Qbar0W, a, b)
    
    return Qbar1W, Qbar0W

# ============================================================================
# X-LEARNER CATE FUNCTIONS FOR EACH METHOD
# ============================================================================

def x_learner_mle_cate(X, t, y, ps):
    """X-learner CATE with MLE outcome modeling (0.5/0.5 weighting)"""
    feature_names = [f'X{i}' for i in range(X.shape[1])]
    W_df = pd.DataFrame(X, columns=feature_names)
    
    outcome_vars = feature_names
    propensity_vars = feature_names
    
    if (t==1).sum() > 0 and len(np.unique(y[t==1])) >= 2:
        try:
            Qbar1W_all, _ = mle_outcome_model(
                W_df, np.ones(len(y)), y,
                outcome_vars, np.ones(len(y))
            )
            mu1_pred = Qbar1W_all
        except:
            model = LogisticRegression(max_iter=1000)
            model.fit(X[t==1], y[t==1])
            mu1_pred = np.clip(model.predict_proba(X)[:, 1], 0, 1)
    else:
        mu1_pred = np.full(len(y), y[t==1].mean() if (t==1).sum() > 0 else y.mean())
        mu1_pred = np.clip(mu1_pred, 0, 1)
    
    if (t==0).sum() > 0 and len(np.unique(y[t==0])) >= 2:
        try:
            _, Qbar0W_all = mle_outcome_model(
                W_df, np.zeros(len(y)), y,
                outcome_vars, np.ones(len(y))
            )
            mu0_pred = Qbar0W_all
        except:
            model = LogisticRegression(max_iter=1000)
            model.fit(X[t==0], y[t==0])
            mu0_pred = np.clip(model.predict_proba(X)[:, 1], 0, 1)
    else:
        mu0_pred = np.full(len(y), y[t==0].mean() if (t==0).sum() > 0 else y.mean())
        mu0_pred = np.clip(mu0_pred, 0, 1)
    
    tau1 = y[t==1] - mu0_pred[t==1]
    tau0 = mu1_pred[t==0] - y[t==0]
    
    base_learner = HistGradientBoostingRegressor(
        max_iter=100,
        random_state=42,
        validation_fraction=0.1,
        n_iter_no_change=5
    )
    
    def fit_or_constant(model, X_fit, y_fit):
        if len(np.unique(y_fit)) < 2:
            from sklearn.dummy import DummyRegressor
            return DummyRegressor(strategy='mean').fit(X_fit, y_fit)
        try:
            return model.fit(X_fit, y_fit)
        except:
            from sklearn.dummy import DummyRegressor
            return DummyRegressor(strategy='mean').fit(X_fit, y_fit)
    
    h1 = fit_or_constant(base_learner.__class__(**base_learner.get_params()), X[t==1], tau1)
    h0 = fit_or_constant(base_learner.__class__(**base_learner.get_params()), X[t==0], tau0)
    
    cate = 0.5 * h1.predict(X) + 0.5 * h0.predict(X)
    
    return cate, mu0_pred

def x_learner_tmle_cate(X, t, y, ps):
    """X-learner CATE with TMLE outcome modeling (0.5/0.5 weighting)"""
    feature_names = [f'X{i}' for i in range(X.shape[1])]
    W_df = pd.DataFrame(X, columns=feature_names)
    
    outcome_vars = feature_names
    propensity_vars = feature_names
    
    if (t==1).sum() > 0 and len(np.unique(y[t==1])) >= 2:
        try:
            Qbar1W_all, _ = tmle_outcome_model(
                W_df, np.ones(len(y)), y,
                outcome_vars, propensity_vars, np.ones(len(y))
            )
            mu1_pred = Qbar1W_all
        except:
            model = LogisticRegression(max_iter=1000)
            model.fit(X[t==1], y[t==1])
            mu1_pred = np.clip(model.predict_proba(X)[:, 1], 0, 1)
    else:
        mu1_pred = np.full(len(y), y[t==1].mean() if (t==1).sum() > 0 else y.mean())
        mu1_pred = np.clip(mu1_pred, 0, 1)
    
    if (t==0).sum() > 0 and len(np.unique(y[t==0])) >= 2:
        try:
            _, Qbar0W_all = tmle_outcome_model(
                W_df, np.zeros(len(y)), y,
                outcome_vars, propensity_vars, np.ones(len(y))
            )
            mu0_pred = Qbar0W_all
        except:
            model = LogisticRegression(max_iter=1000)
            model.fit(X[t==0], y[t==0])
            mu0_pred = np.clip(model.predict_proba(X)[:, 1], 0, 1)
    else:
        mu0_pred = np.full(len(y), y[t==0].mean() if (t==0).sum() > 0 else y.mean())
        mu0_pred = np.clip(mu0_pred, 0, 1)
    
    tau1 = y[t==1] - mu0_pred[t==1]
    tau0 = mu1_pred[t==0] - y[t==0]
    
    base_learner = HistGradientBoostingRegressor(
        max_iter=100,
        random_state=42,
        validation_fraction=0.1,
        n_iter_no_change=5
    )
    
    def fit_or_constant(model, X_fit, y_fit):
        if len(np.unique(y_fit)) < 2:
            from sklearn.dummy import DummyRegressor
            return DummyRegressor(strategy='mean').fit(X_fit, y_fit)
        try:
            return model.fit(X_fit, y_fit)
        except:
            from sklearn.dummy import DummyRegressor
            return DummyRegressor(strategy='mean').fit(X_fit, y_fit)
    
    h1 = fit_or_constant(base_learner.__class__(**base_learner.get_params()), X[t==1], tau1)
    h0 = fit_or_constant(base_learner.__class__(**base_learner.get_params()), X[t==0], tau0)
    
    cate = 0.5 * h1.predict(X) + 0.5 * h0.predict(X)
    
    return cate, mu0_pred

def x_learner_rtmle_cate(X, t, y, ps, a=None, b=None):
    """
    X-learner CATE with rTMLE outcome modeling (0.5/0.5 weighting)
    
    Parameters:
    -----------
    a : float, optional
        Lower bound for rTMLE. If None, defaults to 0
    b : float, optional
        Upper bound for rTMLE. If None, uses paper310 formula: min(0.5, 7 * mean(Y))
    """
    feature_names = [f'X{i}' for i in range(X.shape[1])]
    W_df = pd.DataFrame(X, columns=feature_names)
    
    outcome_vars = feature_names
    propensity_vars = feature_names
    
    # ========================================================================
    # PROPER BOUNDS SELECTION (Paper310/Balzer style)
    # ========================================================================
    if a is None:
        a = 0.0
    
    if b is None:
        outcome_prevalence = y.mean()
        # Paper310 recommendation: b = min(0.5, 7 * mean(Y))
        b = min(0.5, 7 * outcome_prevalence)
        b = max(b, 0.01)  # Ensure minimum bound
    
    if (t==1).sum() > 0 and len(np.unique(y[t==1])) >= 2:
        try:
            Qbar1W_all, _ = rtmle_outcome_model(
                W_df, np.ones(len(y)), y, a, b,
                outcome_vars, propensity_vars, np.ones(len(y))
            )
            mu1_pred = Qbar1W_all
        except:
            model = LogisticRegression(max_iter=1000)
            model.fit(X[t==1], y[t==1])
            mu1_pred = np.clip(model.predict_proba(X)[:, 1], a, b)
    else:
        mu1_pred = np.full(len(y), y[t==1].mean() if (t==1).sum() > 0 else y.mean())
        mu1_pred = np.clip(mu1_pred, a, b)
    
    if (t==0).sum() > 0 and len(np.unique(y[t==0])) >= 2:
        try:
            _, Qbar0W_all = rtmle_outcome_model(
                W_df, np.zeros(len(y)), y, a, b,
                outcome_vars, propensity_vars, np.ones(len(y))
            )
            mu0_pred = Qbar0W_all
        except:
            model = LogisticRegression(max_iter=1000)
            model.fit(X[t==0], y[t==0])
            mu0_pred = np.clip(model.predict_proba(X)[:, 1], a, b)
    else:
        mu0_pred = np.full(len(y), y[t==0].mean() if (t==0).sum() > 0 else y.mean())
        mu0_pred = np.clip(mu0_pred, a, b)
    
    tau1 = y[t==1] - mu0_pred[t==1]
    tau0 = mu1_pred[t==0] - y[t==0]
    
    base_learner = HistGradientBoostingRegressor(
        max_iter=100,
        random_state=42,
        validation_fraction=0.1,
        n_iter_no_change=5
    )
    
    def fit_or_constant(model, X_fit, y_fit):
        if len(np.unique(y_fit)) < 2:
            from sklearn.dummy import DummyRegressor
            return DummyRegressor(strategy='mean').fit(X_fit, y_fit)
        try:
            return model.fit(X_fit, y_fit)
        except:
            from sklearn.dummy import DummyRegressor
            return DummyRegressor(strategy='mean').fit(X_fit, y_fit)
    
    h1 = fit_or_constant(base_learner.__class__(**base_learner.get_params()), X[t==1], tau1)
    h0 = fit_or_constant(base_learner.__class__(**base_learner.get_params()), X[t==0], tau0)
    
    cate = 0.5 * h1.predict(X) + 0.5 * h0.predict(X)
    
    return cate, mu0_pred


def compute_gate(cate, mu0, n_quantiles=4):
    """Compute GATEs by baseline risk quartiles"""
    try:
        percentiles = np.percentile(mu0, [25, 50, 75])
        gate_rows = []
        
        mask1 = mu0 < percentiles[0]
        if mask1.sum() > 0:
            gate_rows.append({
                'Quartile': 'Q1_Low',
                'GATE': cate[mask1].mean(),
                'n': mask1.sum(),
                'mean_baseline_risk': mu0[mask1].mean()
            })
        
        mask2 = (mu0 >= percentiles[0]) & (mu0 < percentiles[1])
        if mask2.sum() > 0:
            gate_rows.append({
                'Quartile': 'Q2_Med-Low',
                'GATE': cate[mask2].mean(),
                'n': mask2.sum(),
                'mean_baseline_risk': mu0[mask2].mean()
            })
        
        mask3 = (mu0 >= percentiles[1]) & (mu0 < percentiles[2])
        if mask3.sum() > 0:
            gate_rows.append({
                'Quartile': 'Q3_Med-High',
                'GATE': cate[mask3].mean(),
                'n': mask3.sum(),
                'mean_baseline_risk': mu0[mask3].mean()
            })
        
        mask4 = mu0 >= percentiles[2]
        if mask4.sum() > 0:
            gate_rows.append({
                'Quartile': 'Q4_High',
                'GATE': cate[mask4].mean(),
                'n': mask4.sum(),
                'mean_baseline_risk': mu0[mask4].mean()
            })
        
        if len(gate_rows) > 0:
            return pd.DataFrame(gate_rows)
        else:
            return pd.DataFrame()
    except Exception as e:
        return pd.DataFrame()

def bootstrap_gate_ci(cate, mu0, n_quantiles=4, n_reps=1000, random_state=42):
    """Bootstrap confidence intervals for GATEs"""
    rng = np.random.default_rng(random_state)
    n = len(cate)
    
    gate_boot = []
    
    for _ in tqdm(range(n_reps), desc="      Bootstrap CIs", leave=False):
        idx = rng.integers(0, n, n)
        cate_boot = cate[idx]
        mu0_boot = mu0[idx]
        
        try:
            percentiles = np.percentile(mu0_boot, [25, 50, 75])
            
            mask1 = mu0_boot < percentiles[0]
            if mask1.sum() > 0:
                gate_boot.append({'quartile': 0, 'gate': cate_boot[mask1].mean()})
            
            mask2 = (mu0_boot >= percentiles[0]) & (mu0_boot < percentiles[1])
            if mask2.sum() > 0:
                gate_boot.append({'quartile': 1, 'gate': cate_boot[mask2].mean()})
            
            mask3 = (mu0_boot >= percentiles[1]) & (mu0_boot < percentiles[2])
            if mask3.sum() > 0:
                gate_boot.append({'quartile': 2, 'gate': cate_boot[mask3].mean()})
            
            mask4 = mu0_boot >= percentiles[2]
            if mask4.sum() > 0:
                gate_boot.append({'quartile': 3, 'gate': cate_boot[mask4].mean()})
        except:
            continue
    
    if len(gate_boot) == 0:
        return {}
    
    gate_boot_df = pd.DataFrame(gate_boot)
    
    ci_dict = {}
    for q in range(n_quantiles):
        q_gates = gate_boot_df[gate_boot_df['quartile'] == q]['gate'].values
        if len(q_gates) > 0:
            ci_dict[q] = {
                'low': np.percentile(q_gates, 2.5),
                'high': np.percentile(q_gates, 97.5)
            }
    
    return ci_dict

# ============================================================================
# MAIN ANALYSIS: COMPARE MLE, TMLE, rTMLE FOR EACH SIGNIFICANT DRUG
# ============================================================================

# Get significant drugs from ATE results
significant_drugs = results_df[results_df['Significant'] == True]['Treatment'].tolist()

print("=" * 100)
print("INDIVIDUAL DRUG GATE ANALYSIS: MLE vs TMLE vs rTMLE COMPARISON")
print("=" * 100)
print(f"Analyzing {len(significant_drugs)} significant drugs\n")

all_comparison_results = []

for idx, drug in enumerate(significant_drugs, 1):
    print("\n" + "=" * 100)
    print(f"Drug {idx}/{len(significant_drugs)}: {drug}")
    print("=" * 100)
    
    if drug not in balanced_samples:
        print(f"  [SKIP] No balanced sample available")
        continue
    
    drug_name = balanced_samples[drug]['drug_name']
    print(f"  {drug_name}")
    
    # Get balanced indices and create dataframe
    balanced_idx = balanced_samples[drug]['balanced_indices']
    df_balanced = tx.loc[balanced_idx].copy()
    
    t = df_balanced[drug].astype(int).values
    y = df_balanced['overdose'].astype(int).values
    
    print(f"  Balanced sample: {t.sum()} treated, {len(t) - t.sum()} control")
    print(f"  Outcome: {y.sum()} positive, {len(y) - y.sum()} negative")
    
    # Prepare covariates
    drop_cols = ['person_id', 'index_date', drug, 'overdose']
    drop_cols = [c for c in drop_cols if c in df_balanced.columns]
    
    feature_cols = [c for c in df_balanced.columns 
                    if c not in drop_cols 
                    and df_balanced[c].dtype != 'O']
    
    X = df_balanced[feature_cols].fillna(0.0).astype("float32").values
    
    print(f"  Features: {len(feature_cols)}, Patients: {len(X)}")
    
    if t.sum() == 0 or t.sum() == len(t):
        print(f"  [SKIP] No variation in treatment")
        continue
    
    # Estimate outcome prevalence for rTMLE bounds (Paper310 formula)
    outcome_prevalence = y.mean()
    # Paper310 recommendation: b = min(0.5, 7 * mean(Y))
    a_rtmle = 0.0
    b_rtmle = min(0.5, 7 * outcome_prevalence)
    b_rtmle = max(b_rtmle, 0.01)  # Ensure minimum bound
    
    print(f"  Outcome prevalence: {outcome_prevalence:.4f}")
    print(f"  rTMLE bounds: a={a_rtmle:.4f}, b={b_rtmle:.4f} (Paper310 formula)")
    
    # Propensity score
    print(f"\n  [1] Estimating propensity scores...")
    ps_model = LogisticRegression(max_iter=1000, random_state=42)
    ps = ps_model.fit(X, t).predict_proba(X)[:, 1]
    ps = np.clip(ps, 1e-6, 1 - 1e-6)
    
    ps_auc = roc_auc_score(t, ps)
    print(f"      PS AUC: {ps_auc:.3f}")
    
    # Overlap trimming
    print(f"  [2] Applying overlap trimming...")
    keep = (ps >= 0.05) & (ps <= 0.95)
    
    if keep.sum() == 0 or t[keep].sum() == 0 or t[keep].sum() == keep.sum():
        print(f"      [SKIP] Insufficient overlap after trimming")
        continue
    
    X_overlap = X[keep]
    t_overlap = t[keep]
    y_overlap = y[keep]
    ps_overlap = ps[keep]
    
    print(f"      Kept: {keep.sum()}/{len(t)} ({keep.mean():.1%})")
    print(f"      Treated: {t_overlap.sum()}, Control: {(t_overlap==0).sum()}")
    
    # Recalculate bounds after overlap trimming (use trimmed outcome prevalence)
    outcome_prevalence_trimmed = y_overlap.mean()
    b_rtmle_trimmed = min(0.5, 7 * outcome_prevalence_trimmed)
    b_rtmle_trimmed = max(b_rtmle_trimmed, 0.01)
    
    # Run all three methods
    methods_results = {}
    
    for method_name, cate_func in [
        ('MLE', lambda: x_learner_mle_cate(X_overlap, t_overlap, y_overlap, ps_overlap)),
        ('TMLE', lambda: x_learner_tmle_cate(X_overlap, t_overlap, y_overlap, ps_overlap)),
        ('rTMLE', lambda: x_learner_rtmle_cate(X_overlap, t_overlap, y_overlap, ps_overlap, 
                                               a=a_rtmle, b=b_rtmle_trimmed))
    ]:
        print(f"\n  [3] Running {method_name}-GATE...")
        try:
            cate, mu0 = cate_func()
            ate_overall = cate.mean()
            print(f"      Overall ATE: {ate_overall:.4f}")
            
            # Compute GATEs
            gate_df = compute_gate(cate, mu0, n_quantiles=4)
            
            if gate_df.empty:
                print(f"      [SKIP] Could not compute GATEs")
                continue
            
            # Bootstrap CIs
            ci_dict = bootstrap_gate_ci(cate, mu0, n_quantiles=4, n_reps=1000, random_state=42)
            
            # Merge CIs
            quartile_map = {"Q1_Low": 0, "Q2_Med-Low": 1, "Q3_Med-High": 2, "Q4_High": 3}
            for idx_row, row in gate_df.iterrows():
                q_label = row['Quartile']
                q_num = quartile_map.get(q_label, -1)
                if q_num >= 0 and q_num in ci_dict:
                    gate_df.loc[idx_row, 'CI_low'] = ci_dict[q_num]['low']
                    gate_df.loc[idx_row, 'CI_high'] = ci_dict[q_num]['high']
                else:
                    gate_df.loc[idx_row, 'CI_low'] = np.nan
                    gate_df.loc[idx_row, 'CI_high'] = np.nan
            
            gate_df['Method'] = method_name
            gate_df['ATE_Overall'] = ate_overall
            methods_results[method_name] = gate_df
            
            print(f"      GATEs computed: {len(gate_df)} quartiles")
            
        except Exception as e:
            print(f"      [ERROR] {method_name}-GATE failed: {str(e)}")
            import traceback
            traceback.print_exc()
            continue
    
    # Combine results for this drug
    if len(methods_results) > 0:
        combined_df = pd.concat(methods_results.values(), ignore_index=True)
        combined_df['Drug'] = drug
        combined_df['Drug_Name'] = drug_name
        combined_df['Treatment'] = drug
        combined_df['Outcome'] = 'overdose'
        combined_df['N'] = len(X_overlap)
        combined_df['Upper_Bound'] = b_rtmle_trimmed if 'rTMLE' in methods_results else np.nan
        combined_df['Lower_Bound'] = a_rtmle if 'rTMLE' in methods_results else np.nan
        
        # Reorder columns
        col_order = ['Drug', 'Drug_Name', 'Treatment', 'Outcome', 'N', 'Method', 
                     'ATE_Overall', 'Lower_Bound', 'Upper_Bound', 'Quartile', 'n', 'mean_baseline_risk', 
                     'GATE', 'CI_low', 'CI_high']
        combined_df = combined_df[[c for c in col_order if c in combined_df.columns]]
        
        all_comparison_results.append(combined_df)
        
        # Print comparison
        print(f"\n  COMPARISON SUMMARY:")
        print(combined_df[['Method', 'Quartile', 'GATE', 'CI_low', 'CI_high']].to_string(index=False))

# Combine all results
if len(all_comparison_results) > 0:
    all_comparison_results_df = pd.concat(all_comparison_results, ignore_index=True)
    
    print("\n" + "=" * 100)
    print("SUMMARY: ALL MLE/TMLE/rTMLE-GATE COMPARISON RESULTS")
    print("=" * 100)
    print(all_comparison_results_df.to_string(index=False))
    
    # Save results
    all_comparison_results_df.to_csv('gate_results_mle_tmle_rtmle_comparison_opioid.csv', index=False)
    print(f"\nResults saved to 'gate_results_mle_tmle_rtmle_comparison_opioid.csv'")
    
    # Create side-by-side comparison table
    print("\n" + "=" * 100)
    print("SIDE-BY-SIDE COMPARISON BY DRUG AND QUARTILE")
    print("=" * 100)
    
    comparison_pivot = all_comparison_results_df.pivot_table(
        index=['Drug_Name', 'Quartile'],
        columns='Method',
        values=['GATE', 'CI_low', 'CI_high'],
        aggfunc='first'
    )
    
    print(comparison_pivot.to_string())
    
    # Summary statistics
    print("\n" + "=" * 100)
    print("METHOD COMPARISON SUMMARY")
    print("=" * 100)
    
    for method in ['MLE', 'TMLE', 'rTMLE']:
        method_df = all_comparison_results_df[all_comparison_results_df['Method'] == method]
        if len(method_df) > 0:
            significant = method_df[
                (~method_df['CI_low'].isna()) & 
                ((method_df['CI_low'] > 0) | (method_df['CI_high'] < 0))
            ]
            print(f"\n{method}-GATE:")
            print(f"  Total GATEs computed: {len(method_df)}")
            print(f"  Significant effects: {len(significant)}")
            print(f"  Mean |GATE|: {method_df['GATE'].abs().mean():.4f}")
            # Filter out NaN CIs for width calculation
            valid_cis = method_df[~method_df['CI_low'].isna()]
            if len(valid_cis) > 0:
                print(f"  Mean CI width: {(valid_cis['CI_high'] - valid_cis['CI_low']).mean():.4f}")
            else:
                print(f"  Mean CI width: N/A (no valid CIs)")
else:
    print("\nNo comparison results generated.")

INDIVIDUAL DRUG GATE ANALYSIS: MLE vs TMLE vs rTMLE COMPARISON
Analyzing 4 significant drugs


Drug 1/4: drug_1110410
  morphine
  Balanced sample: 855 treated, 855 control
  Outcome: 10 positive, 1700 negative
  Features: 2566, Patients: 1710
  Outcome prevalence: 0.0058
  rTMLE bounds: a=0.0000, b=0.0409 (Paper310 formula)

  [1] Estimating propensity scores...
      PS AUC: 0.919
  [2] Applying overlap trimming...
      Kept: 1464/1710 (85.6%)
      Treated: 682, Control: 782

  [3] Running MLE-GATE...
      Overall ATE: 0.0029


      GATEs computed: 4 quartiles

  [3] Running TMLE-GATE...
      Overall ATE: 0.0080


      GATEs computed: 4 quartiles

  [3] Running rTMLE-GATE...
      Overall ATE: 0.0063


      GATEs computed: 4 quartiles

  COMPARISON SUMMARY:
Method    Quartile     GATE    CI_low  CI_high
   MLE      Q1_Low 0.000564 -0.000057 0.001249
   MLE  Q2_Med-Low 0.000127 -0.000480 0.000837
   MLE Q3_Med-High 0.001853  0.000599 0.002869
   MLE     Q4_High 0.009246  0.007150 0.011732
  TMLE      Q1_Low 0.015031  0.011038 0.019963
  TMLE  Q2_Med-Low 0.004813  0.003362 0.006581
  TMLE Q3_Med-High 0.007081  0.005435 0.009105
  TMLE     Q4_High 0.004979  0.002918 0.007148
 rTMLE      Q1_Low 0.009007  0.006108 0.012674
 rTMLE  Q2_Med-Low 0.004556  0.003163 0.006183
 rTMLE Q3_Med-High 0.006703  0.005058 0.008653
 rTMLE     Q4_High 0.005049  0.003073 0.007167

Drug 2/4: drug_19134047
  tramadol hydrochloride 50 MG Oral Tablet
  Balanced sample: 1052 treated, 1052 control
  Outcome: 13 positive, 2091 negative
  Features: 2566, Patients: 2104
  Outcome prevalence: 0.0062
  rTMLE bounds: a=0.0000, b=0.0433 (Paper310 formula)

  [1] Estimating propensity scores...
      PS AUC: 0.876
  [2]

      GATEs computed: 4 quartiles

  [3] Running TMLE-GATE...
      Overall ATE: 0.0070


...

      GATEs computed: 4 quartiles

  [3] Running rTMLE-GATE...
      Overall ATE: 0.0251


      GATEs computed: 4 quartiles

  COMPARISON SUMMARY:
Method    Quartile      GATE    CI_low   CI_high
   MLE      Q1_Low -0.002316 -0.003148 -0.001180
   MLE  Q2_Med-Low  0.000992 -0.000459  0.003173
   MLE Q3_Med-High  0.009695  0.006298  0.012671
   MLE     Q4_High  0.021494  0.016806  0.026719
  TMLE      Q1_Low  0.003150  0.000619  0.012632
  TMLE  Q2_Med-Low  0.030290  0.021243  0.040835
  TMLE Q3_Med-High  0.047005  0.034970  0.056312
  TMLE     Q4_High  0.049258  0.034027  0.066865
 rTMLE      Q1_Low  0.002874  0.000479  0.011913
 rTMLE  Q2_Med-Low  0.028366  0.019819  0.038136
 rTMLE Q3_Med-High  0.043552  0.032279  0.052757
 rTMLE     Q4_High  0.025799  0.018059  0.034912

Drug 4/4: drug_40223140
  1 ML morphine sulfate 2 MG/ML Prefilled Syringe
  Balanced sample: 1143 treated, 1143 control
  Outcome: 17 positive, 2269 negative
  Features: 2566, Patients: 2286
  Outcome prevalence: 0.0074
  rTMLE bounds: a=0.0000, b=0.0521 (Paper310 formula)

  [1] Estimating propensity sc

      GATEs computed: 4 quartiles

  [3] Running TMLE-GATE...
      Overall ATE: 0.0067


      GATEs computed: 4 quartiles

  [3] Running rTMLE-GATE...
      Overall ATE: 0.0063


      GATEs computed: 4 quartiles

  COMPARISON SUMMARY:
Method    Quartile      GATE    CI_low   CI_high
   MLE      Q1_Low -0.000913 -0.001727 -0.000157
   MLE  Q2_Med-Low -0.000549 -0.001184  0.000371
   MLE Q3_Med-High  0.001477  0.000355  0.002493
   MLE     Q4_High  0.007333  0.005119  0.009629
  TMLE      Q1_Low  0.009288  0.007199  0.011435
  TMLE  Q2_Med-Low  0.006395  0.005045  0.007874
  TMLE Q3_Med-High  0.007922  0.006003  0.009882
  TMLE     Q4_High  0.003393  0.000802  0.006031
 rTMLE      Q1_Low  0.008766  0.006730  0.010835
 rTMLE  Q2_Med-Low  0.005888  0.004601  0.007336
 rTMLE Q3_Med-High  0.007969  0.006056  0.009904
 rTMLE     Q4_High  0.002643  0.000043  0.005206

SUMMARY: ALL MLE/TMLE/rTMLE-GATE COMPARISON RESULTS
         Drug                                           Drug_Name     Treatment  Outcome    N Method  ATE_Overall  Lower_Bound  Upper_Bound    Quartile   n  mean_baseline_risk      GATE    CI_low   CI_high
 drug_1110410                                  

# rTMLE-GATE_PAIRWISE

In [54]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import roc_auc_score
from scipy.optimize import minimize
from scipy.special import expit, logit
from tqdm import tqdm
from itertools import combinations
import warnings
warnings.filterwarnings('ignore')


def log_like(beta, Y, X, wt=None):
    if wt is None:
        wt = np.ones(len(Y))
    pi = expit(X @ beta)
    pi = np.clip(pi, np.finfo(float).eps, 1 - np.finfo(float).eps)
    log_like = np.sum(wt * (Y * np.log(pi) + (1 - Y) * np.log(1 - pi)))
    return -log_like

def grad(beta, Y, X, wt=None):
    if wt is None:
        wt = np.ones(len(Y))
    pi = expit(X @ beta)
    pi = np.clip(pi, np.finfo(float).eps, 1 - np.finfo(float).eps)
    resid = wt * (Y - pi)
    gr = X.T @ resid
    return -gr

def tmle_log_like(eps, Y, Q_AW, H_0W, H_1W, wt=None):
    if wt is None:
        wt = np.ones(len(Y))
    pi = expit(logit(Q_AW) + eps[0] * H_0W + eps[1] * H_1W)
    pi = np.clip(pi, np.finfo(float).eps, 1 - np.finfo(float).eps)
    log_like = np.sum(wt * (Y * np.log(pi) + (1 - Y) * np.log(1 - pi)))
    if np.isnan(log_like) or np.isinf(log_like):
        return 99
    return -log_like

def tmle_grad(eps, Y, Q_AW, H_0W, H_1W, wt=None):
    if wt is None:
        wt = np.ones(len(Y))
    pi = expit(logit(Q_AW) + eps[0] * H_0W + eps[1] * H_1W)
    pi = np.clip(pi, np.finfo(float).eps, 1 - np.finfo(float).eps)
    resid = wt * (Y - pi)
    gr = np.array([np.sum(H_0W * resid), np.sum(H_1W * resid)])
    if np.any(np.isnan(gr)) or np.any(np.isinf(gr)):
        return np.array([99, 99])
    return -gr

# ============================================================================
# OUTCOME MODELING FUNCTIONS: MLE, TMLE, rTMLE
# ============================================================================

def mle_outcome_model(W, A, Y, outcome_vars, weights=None):
    """MLE: Simple logistic regression for outcome modeling"""
    if weights is None:
        weights = np.ones(len(Y))
    
    data_temp = pd.DataFrame(W)
    data_temp['A'] = A
    
    X_model = data_temp[outcome_vars + ['A']].values
    X_model = np.column_stack([np.ones(len(X_model)), X_model])
    
    try:
        model = LogisticRegression(max_iter=1000, fit_intercept=False)
        model.fit(X_model, Y, sample_weight=weights)
        beta = model.coef_[0]
    except:
        try:
            result = minimize(
                log_like,
                x0=np.zeros(X_model.shape[1]),
                args=(Y, X_model, weights),
                method='BFGS',
                jac=grad,
                options={'maxiter': 500}
            )
            beta = result.x
        except:
            beta = np.zeros(X_model.shape[1])
    
    X1 = X_model.copy()
    X0 = X_model.copy()
    A_idx = len(outcome_vars) + 1
    X1[:, A_idx] = 1
    X0[:, A_idx] = 0
    
    Qbar1W = expit(X1 @ beta)
    Qbar0W = expit(X0 @ beta)
    
    return Qbar1W, Qbar0W

def tmle_outcome_model(W, A, Y, outcome_vars, propensity_vars, weights=None):
    """TMLE: Targeted Maximum Likelihood Estimation for outcome modeling"""
    if weights is None:
        weights = np.ones(len(Y))
    
    data_temp = pd.DataFrame(W)
    data_temp['A'] = A
    
    X_model = data_temp[outcome_vars + ['A']].values
    X_model = np.column_stack([np.ones(len(X_model)), X_model])
    
    try:
        result = minimize(
            log_like,
            x0=np.zeros(X_model.shape[1]),
            args=(Y, X_model, weights),
            method='BFGS',
            jac=grad,
            options={'maxiter': 500}
        )
        beta = result.x
    except:
        model = LogisticRegression(max_iter=1000, fit_intercept=False)
        model.fit(X_model, Y, sample_weight=weights)
        beta = model.coef_[0]
    
    X1 = X_model.copy()
    X0 = X_model.copy()
    A_idx = len(outcome_vars) + 1
    X1[:, A_idx] = 1
    X0[:, A_idx] = 0
    
    QbarAW = expit(X_model @ beta)
    Qbar1W = expit(X1 @ beta)
    Qbar0W = expit(X0 @ beta)
    
    # Propensity score
    X_g = W[propensity_vars].values
    g_model = LogisticRegression(fit_intercept=True, max_iter=1000)
    g_model.fit(X_g, A, sample_weight=weights)
    g1W = g_model.predict_proba(X_g)[:, 1]
    g0W = 1 - g1W
    
    H_1W = A / np.clip(g1W, 1e-6, 1-1e-6)
    H_0W = (1 - A) / np.clip(g0W, 1e-6, 1-1e-6)
    
    # Targeting step
    try:
        result_eps = minimize(
            tmle_log_like,
            x0=[0, 0],
            args=(Y, QbarAW, H_0W, H_1W, weights),
            method='BFGS',
            jac=tmle_grad,
            options={'maxiter': 500}
        )
        eps = result_eps.x
    except:
        eps = [0, 0]
    
    QbarAW_updated = expit(logit(QbarAW) + eps[0] * H_0W + eps[1] * H_1W)
    Qbar0W_updated = expit(logit(Qbar0W) + eps[0] / np.clip(g0W, 1e-6, 1-1e-6))
    Qbar1W_updated = expit(logit(Qbar1W) + eps[1] / np.clip(g1W, 1e-6, 1-1e-6))
    
    return Qbar1W_updated, Qbar0W_updated

def rtmle_outcome_model(W, A, Y, a, b, outcome_vars, propensity_vars, weights=None):
    """rTMLE: Rare outcomes Targeted Maximum Likelihood Estimation"""
    if weights is None:
        weights = np.ones(len(Y))
    
    Y_tilde = (Y - a) / (b - a)
    
    data_temp = pd.DataFrame(W)
    data_temp['A'] = A
    
    X_model = data_temp[outcome_vars + ['A']].values
    X_model = np.column_stack([np.ones(len(X_model)), X_model])
    
    try:
        result = minimize(
            log_like,
            x0=np.zeros(X_model.shape[1]),
            args=(Y_tilde, X_model, weights),
            method='BFGS',
            jac=grad,
            options={'maxiter': 500}
        )
        beta = result.x
    except:
        model = LogisticRegression(max_iter=1000, fit_intercept=False)
        model.fit(X_model, Y_tilde, sample_weight=weights)
        beta = model.coef_[0]
    
    X1 = X_model.copy()
    X0 = X_model.copy()
    A_idx = len(outcome_vars) + 1
    X1[:, A_idx] = 1
    X0[:, A_idx] = 0
    
    QbarAW = expit(X_model @ beta)
    Qbar1W_tilde = expit(X1 @ beta)
    Qbar0W_tilde = expit(X0 @ beta)
    
    X_g = W[propensity_vars].values
    g_model = LogisticRegression(fit_intercept=True, max_iter=1000)
    g_model.fit(X_g, A, sample_weight=weights)
    g1W = g_model.predict_proba(X_g)[:, 1]
    g0W = 1 - g1W
    
    H_1W = A / np.clip(g1W, 1e-6, 1-1e-6)
    H_0W = (1 - A) / np.clip(g0W, 1e-6, 1-1e-6)
    
    try:
        result_eps = minimize(
            tmle_log_like,
            x0=[0, 0],
            args=(Y_tilde, QbarAW, H_0W, H_1W, weights),
            method='BFGS',
            jac=tmle_grad,
            options={'maxiter': 500}
        )
        eps = result_eps.x
    except:
        eps = [0, 0]
    
    QbarAW_updated = expit(logit(QbarAW) + eps[0] * H_0W + eps[1] * H_1W)
    Qbar0W_updated = expit(logit(Qbar0W_tilde) + eps[0] / np.clip(g0W, 1e-6, 1-1e-6))
    Qbar1W_updated = expit(logit(Qbar1W_tilde) + eps[1] / np.clip(g1W, 1e-6, 1-1e-6))
    
    Qbar1W = Qbar1W_updated * (b - a) + a
    Qbar0W = Qbar0W_updated * (b - a) + a
    
    Qbar1W = np.clip(Qbar1W, a, b)
    Qbar0W = np.clip(Qbar0W, a, b)
    
    return Qbar1W, Qbar0W

# ============================================================================
# X-LEARNER CATE FUNCTIONS FOR EACH METHOD
# ============================================================================

def x_learner_mle_cate(X, t, y, ps):
    """X-learner CATE with MLE outcome modeling (0.5/0.5 weighting)"""
    feature_names = [f'X{i}' for i in range(X.shape[1])]
    W_df = pd.DataFrame(X, columns=feature_names)
    
    outcome_vars = feature_names
    propensity_vars = feature_names
    
    if (t==1).sum() > 0 and len(np.unique(y[t==1])) >= 2:
        try:
            Qbar1W_all, _ = mle_outcome_model(
                W_df, np.ones(len(y)), y,
                outcome_vars, np.ones(len(y))
            )
            mu1_pred = Qbar1W_all
        except:
            model = LogisticRegression(max_iter=1000)
            model.fit(X[t==1], y[t==1])
            mu1_pred = np.clip(model.predict_proba(X)[:, 1], 0, 1)
    else:
        mu1_pred = np.full(len(y), y[t==1].mean() if (t==1).sum() > 0 else y.mean())
        mu1_pred = np.clip(mu1_pred, 0, 1)
    
    if (t==0).sum() > 0 and len(np.unique(y[t==0])) >= 2:
        try:
            _, Qbar0W_all = mle_outcome_model(
                W_df, np.zeros(len(y)), y,
                outcome_vars, np.ones(len(y))
            )
            mu0_pred = Qbar0W_all
        except:
            model = LogisticRegression(max_iter=1000)
            model.fit(X[t==0], y[t==0])
            mu0_pred = np.clip(model.predict_proba(X)[:, 1], 0, 1)
    else:
        mu0_pred = np.full(len(y), y[t==0].mean() if (t==0).sum() > 0 else y.mean())
        mu0_pred = np.clip(mu0_pred, 0, 1)
    
    tau1 = y[t==1] - mu0_pred[t==1]
    tau0 = mu1_pred[t==0] - y[t==0]
    
    base_learner = HistGradientBoostingRegressor(
        max_iter=100,
        random_state=42,
        validation_fraction=0.1,
        n_iter_no_change=5
    )
    
    def fit_or_constant(model, X_fit, y_fit):
        if len(np.unique(y_fit)) < 2:
            from sklearn.dummy import DummyRegressor
            return DummyRegressor(strategy='mean').fit(X_fit, y_fit)
        try:
            return model.fit(X_fit, y_fit)
        except:
            from sklearn.dummy import DummyRegressor
            return DummyRegressor(strategy='mean').fit(X_fit, y_fit)
    
    h1 = fit_or_constant(base_learner.__class__(**base_learner.get_params()), X[t==1], tau1)
    h0 = fit_or_constant(base_learner.__class__(**base_learner.get_params()), X[t==0], tau0)
    
    cate = 0.5 * h1.predict(X) + 0.5 * h0.predict(X)
    
    return cate, mu0_pred

def x_learner_tmle_cate(X, t, y, ps):
    """X-learner CATE with TMLE outcome modeling (0.5/0.5 weighting)"""
    feature_names = [f'X{i}' for i in range(X.shape[1])]
    W_df = pd.DataFrame(X, columns=feature_names)
    
    outcome_vars = feature_names
    propensity_vars = feature_names
    
    if (t==1).sum() > 0 and len(np.unique(y[t==1])) >= 2:
        try:
            Qbar1W_all, _ = tmle_outcome_model(
                W_df, np.ones(len(y)), y,
                outcome_vars, propensity_vars, np.ones(len(y))
            )
            mu1_pred = Qbar1W_all
        except:
            model = LogisticRegression(max_iter=1000)
            model.fit(X[t==1], y[t==1])
            mu1_pred = np.clip(model.predict_proba(X)[:, 1], 0, 1)
    else:
        mu1_pred = np.full(len(y), y[t==1].mean() if (t==1).sum() > 0 else y.mean())
        mu1_pred = np.clip(mu1_pred, 0, 1)
    
    if (t==0).sum() > 0 and len(np.unique(y[t==0])) >= 2:
        try:
            _, Qbar0W_all = tmle_outcome_model(
                W_df, np.zeros(len(y)), y,
                outcome_vars, propensity_vars, np.ones(len(y))
            )
            mu0_pred = Qbar0W_all
        except:
            model = LogisticRegression(max_iter=1000)
            model.fit(X[t==0], y[t==0])
            mu0_pred = np.clip(model.predict_proba(X)[:, 1], 0, 1)
    else:
        mu0_pred = np.full(len(y), y[t==0].mean() if (t==0).sum() > 0 else y.mean())
        mu0_pred = np.clip(mu0_pred, 0, 1)
    
    tau1 = y[t==1] - mu0_pred[t==1]
    tau0 = mu1_pred[t==0] - y[t==0]
    
    base_learner = HistGradientBoostingRegressor(
        max_iter=100,
        random_state=42,
        validation_fraction=0.1,
        n_iter_no_change=5
    )
    
    def fit_or_constant(model, X_fit, y_fit):
        if len(np.unique(y_fit)) < 2:
            from sklearn.dummy import DummyRegressor
            return DummyRegressor(strategy='mean').fit(X_fit, y_fit)
        try:
            return model.fit(X_fit, y_fit)
        except:
            from sklearn.dummy import DummyRegressor
            return DummyRegressor(strategy='mean').fit(X_fit, y_fit)
    
    h1 = fit_or_constant(base_learner.__class__(**base_learner.get_params()), X[t==1], tau1)
    h0 = fit_or_constant(base_learner.__class__(**base_learner.get_params()), X[t==0], tau0)
    
    cate = 0.5 * h1.predict(X) + 0.5 * h0.predict(X)
    
    return cate, mu0_pred

def x_learner_rtmle_cate(X, t, y, ps, a=None, b=None):
    """
    X-learner CATE with rTMLE outcome modeling (0.5/0.5 weighting)
    
    Parameters:
    -----------
    a : float, optional
        Lower bound for rTMLE. If None, defaults to 0
    b : float, optional
        Upper bound for rTMLE. If None, uses paper310 formula: min(0.5, 7 * mean(Y))
    """
    feature_names = [f'X{i}' for i in range(X.shape[1])]
    W_df = pd.DataFrame(X, columns=feature_names)
    
    outcome_vars = feature_names
    propensity_vars = feature_names
    
    # ========================================================================
    # PROPER BOUNDS SELECTION (Paper310/Balzer style)
    # ========================================================================
    if a is None:
        a = 0.0
    
    if b is None:
        outcome_prevalence = y.mean()
        # Paper310 recommendation: b = min(0.5, 7 * mean(Y))
        b = min(0.5, 7 * outcome_prevalence)
        b = max(b, 0.01)  # Ensure minimum bound
    
    if (t==1).sum() > 0 and len(np.unique(y[t==1])) >= 2:
        try:
            Qbar1W_all, _ = rtmle_outcome_model(
                W_df, np.ones(len(y)), y, a, b,
                outcome_vars, propensity_vars, np.ones(len(y))
            )
            mu1_pred = Qbar1W_all
        except:
            model = LogisticRegression(max_iter=1000)
            model.fit(X[t==1], y[t==1])
            mu1_pred = np.clip(model.predict_proba(X)[:, 1], a, b)
    else:
        mu1_pred = np.full(len(y), y[t==1].mean() if (t==1).sum() > 0 else y.mean())
        mu1_pred = np.clip(mu1_pred, a, b)
    
    if (t==0).sum() > 0 and len(np.unique(y[t==0])) >= 2:
        try:
            _, Qbar0W_all = rtmle_outcome_model(
                W_df, np.zeros(len(y)), y, a, b,
                outcome_vars, propensity_vars, np.ones(len(y))
            )
            mu0_pred = Qbar0W_all
        except:
            model = LogisticRegression(max_iter=1000)
            model.fit(X[t==0], y[t==0])
            mu0_pred = np.clip(model.predict_proba(X)[:, 1], a, b)
    else:
        mu0_pred = np.full(len(y), y[t==0].mean() if (t==0).sum() > 0 else y.mean())
        mu0_pred = np.clip(mu0_pred, a, b)
    
    tau1 = y[t==1] - mu0_pred[t==1]
    tau0 = mu1_pred[t==0] - y[t==0]
    
    base_learner = HistGradientBoostingRegressor(
        max_iter=100,
        random_state=42,
        validation_fraction=0.1,
        n_iter_no_change=5
    )
    
    def fit_or_constant(model, X_fit, y_fit):
        if len(np.unique(y_fit)) < 2:
            from sklearn.dummy import DummyRegressor
            return DummyRegressor(strategy='mean').fit(X_fit, y_fit)
        try:
            return model.fit(X_fit, y_fit)
        except:
            from sklearn.dummy import DummyRegressor
            return DummyRegressor(strategy='mean').fit(X_fit, y_fit)
    
    h1 = fit_or_constant(base_learner.__class__(**base_learner.get_params()), X[t==1], tau1)
    h0 = fit_or_constant(base_learner.__class__(**base_learner.get_params()), X[t==0], tau0)
    
    cate = 0.5 * h1.predict(X) + 0.5 * h0.predict(X)
    
    return cate, mu0_pred


def compute_gate(cate, mu0, n_quantiles=4):
    """Compute GATEs by baseline risk quartiles"""
    try:
        percentiles = np.percentile(mu0, [25, 50, 75])
        gate_rows = []
        
        mask1 = mu0 < percentiles[0]
        if mask1.sum() > 0:
            gate_rows.append({
                'Quartile': 'Q1_Low',
                'GATE': cate[mask1].mean(),
                'n': mask1.sum(),
                'mean_baseline_risk': mu0[mask1].mean()
            })
        
        mask2 = (mu0 >= percentiles[0]) & (mu0 < percentiles[1])
        if mask2.sum() > 0:
            gate_rows.append({
                'Quartile': 'Q2_Med-Low',
                'GATE': cate[mask2].mean(),
                'n': mask2.sum(),
                'mean_baseline_risk': mu0[mask2].mean()
            })
        
        mask3 = (mu0 >= percentiles[1]) & (mu0 < percentiles[2])
        if mask3.sum() > 0:
            gate_rows.append({
                'Quartile': 'Q3_Med-High',
                'GATE': cate[mask3].mean(),
                'n': mask3.sum(),
                'mean_baseline_risk': mu0[mask3].mean()
            })
        
        mask4 = mu0 >= percentiles[2]
        if mask4.sum() > 0:
            gate_rows.append({
                'Quartile': 'Q4_High',
                'GATE': cate[mask4].mean(),
                'n': mask4.sum(),
                'mean_baseline_risk': mu0[mask4].mean()
            })
        
        if len(gate_rows) > 0:
            return pd.DataFrame(gate_rows)
        else:
            return pd.DataFrame()
    except Exception as e:
        return pd.DataFrame()

def bootstrap_gate_ci(cate, mu0, n_quantiles=4, n_reps=1000, random_state=42):
    """Bootstrap confidence intervals for GATEs"""
    rng = np.random.default_rng(random_state)
    n = len(cate)
    
    gate_boot = []
    
    for _ in tqdm(range(n_reps), desc="      Bootstrap CIs", leave=False):
        idx = rng.integers(0, n, n)
        cate_boot = cate[idx]
        mu0_boot = mu0[idx]
        
        try:
            percentiles = np.percentile(mu0_boot, [25, 50, 75])
            
            mask1 = mu0_boot < percentiles[0]
            if mask1.sum() > 0:
                gate_boot.append({'quartile': 0, 'gate': cate_boot[mask1].mean()})
            
            mask2 = (mu0_boot >= percentiles[0]) & (mu0_boot < percentiles[1])
            if mask2.sum() > 0:
                gate_boot.append({'quartile': 1, 'gate': cate_boot[mask2].mean()})
            
            mask3 = (mu0_boot >= percentiles[1]) & (mu0_boot < percentiles[2])
            if mask3.sum() > 0:
                gate_boot.append({'quartile': 2, 'gate': cate_boot[mask3].mean()})
            
            mask4 = mu0_boot >= percentiles[2]
            if mask4.sum() > 0:
                gate_boot.append({'quartile': 3, 'gate': cate_boot[mask4].mean()})
        except:
            continue
    
    if len(gate_boot) == 0:
        return {}
    
    gate_boot_df = pd.DataFrame(gate_boot)
    
    ci_dict = {}
    for q in range(n_quantiles):
        q_gates = gate_boot_df[gate_boot_df['quartile'] == q]['gate'].values
        if len(q_gates) > 0:
            ci_dict[q] = {
                'low': np.percentile(q_gates, 2.5),
                'high': np.percentile(q_gates, 97.5)
            }
    
    return ci_dict

# ============================================================================
# OPTIONAL: Query detailed exposures for overlap checking
# ============================================================================
# Set to True if you want to enforce overlapping/co-prescription windows
ENFORCE_OVERLAP = False  # Set to True to enable overlap checking
OVERLAP_WINDOW_DAYS = 7  # Days within which exposures must occur to count as "combination"

if ENFORCE_OVERLAP:
    print("Querying detailed exposure dates for overlap checking...")
    sql_pairwise_exposures = f"""
    WITH cohort AS ({sql_index_surgery}),
    opioid_ingredients AS (
      SELECT concept_id
      FROM `{CDR}.concept`
      WHERE concept_class_id = 'Ingredient'
        AND vocabulary_id = 'RxNorm'
        AND LOWER(concept_name) IN (
          'morphine', 'oxycodone', 'hydromorphone', 'fentanyl',
          'codeine', 'tramadol', 'oxymorphone', 'buprenorphine',
          'meperidine', 'methadone'
        )
    ),
    opioid_ingredient_descendants AS (
      SELECT DISTINCT ca.descendant_concept_id AS ingredient_concept_id
      FROM `{CDR}.concept_ancestor` ca
      JOIN opioid_ingredients oi ON ca.ancestor_concept_id = oi.concept_id
    ),
    opioid_drugs AS (
      SELECT DISTINCT ds.drug_concept_id
      FROM `{CDR}.drug_strength` ds
      JOIN opioid_ingredient_descendants oid
        ON ds.ingredient_concept_id = oid.ingredient_concept_id
    )
    SELECT
      c.person_id,
      de.drug_concept_id,
      DATE(de.drug_exposure_start_date) AS exposure_start,
      COALESCE(DATE(de.drug_exposure_end_date), DATE(de.drug_exposure_start_date)) AS exposure_end,
      DATE_DIFF(DATE(de.drug_exposure_start_date), DATE(c.index_date), DAY) AS days_from_surgery
    FROM `{CDR}.drug_exposure` de
    JOIN cohort c ON de.person_id = c.person_id
    JOIN opioid_drugs od ON de.drug_concept_id = od.drug_concept_id
    WHERE DATE(de.drug_exposure_start_date) >= DATE(c.index_date)
      AND DATE_DIFF(DATE(de.drug_exposure_start_date), DATE(c.index_date), DAY) <= 90
    """
    pairwise_exposures = bq(sql_pairwise_exposures)
    print(f"  Loaded {len(pairwise_exposures)} exposure records")
else:
    pairwise_exposures = None
    print("Overlap checking disabled (using 'ever exposed' definition)")

def check_overlap(person_id, drug1_id, drug2_id, exposures_df, window_days=7):
    """Check if person has overlapping exposures to drug1 and drug2 within window_days"""
    if exposures_df is None:
        return False
    
    d1_exposures = exposures_df[
        (exposures_df['person_id'] == person_id) & 
        (exposures_df['drug_concept_id'] == drug1_id)
    ]
    d2_exposures = exposures_df[
        (exposures_df['person_id'] == person_id) & 
        (exposures_df['drug_concept_id'] == drug2_id)
    ]
    
    if len(d1_exposures) == 0 or len(d2_exposures) == 0:
        return False
    
    for _, d1_row in d1_exposures.iterrows():
        d1_start = d1_row['days_from_surgery']
        d1_end = d1_start + (pd.to_datetime(d1_row['exposure_end']) - pd.to_datetime(d1_row['exposure_start'])).days
        
        for _, d2_row in d2_exposures.iterrows():
            d2_start = d2_row['days_from_surgery']
            d2_end = d2_start + (pd.to_datetime(d2_row['exposure_end']) - pd.to_datetime(d2_row['exposure_start'])).days
            
            # Check if ranges overlap or are within window
            if (d1_start <= d2_end + window_days) and (d2_start <= d1_end + window_days):
                return True
    
    return False

# ============================================================================
# MAIN ANALYSIS: COMPARE MLE, TMLE, rTMLE FOR PAIRWISE COMBINATIONS
# ============================================================================

# Get significant drugs
significant_drugs = results_df[results_df['Significant'] == True]['Treatment'].tolist()

# Generate all pairs
drug_pairs = list(combinations(significant_drugs, 2))

# Limit to top pairs if too many (optional)
MAX_PAIRS = 50
if len(drug_pairs) > MAX_PAIRS:
    print(f"Limiting to top {MAX_PAIRS} pairs (out of {len(drug_pairs)} total)")
    drug_pairs = drug_pairs[:MAX_PAIRS]

print("=" * 100)
print("PAIRWISE DRUG GATE ANALYSIS: MLE vs TMLE vs rTMLE COMPARISON")
print("=" * 100)
print(f"Analyzing {len(drug_pairs)} drug pairs from {len(significant_drugs)} significant drugs")
print(f"Control group: NO opioids at all (strict definition)")
if ENFORCE_OVERLAP:
    print(f"Overlap enforcement: Enabled (window = {OVERLAP_WINDOW_DAYS} days)")
else:
    print(f"Overlap enforcement: Disabled (ever exposed definition)")
print()

all_pairwise_comparison_results = []

for idx, (drug1, drug2) in enumerate(drug_pairs, 1):
    print("\n" + "=" * 100)
    print(f"Pair {idx}/{len(drug_pairs)}: {drug1} + {drug2}")
    print("=" * 100)
    
    drug1_name = balanced_samples[drug1]['drug_name'] if drug1 in balanced_samples else drug1
    drug2_name = balanced_samples[drug2]['drug_name'] if drug2 in balanced_samples else drug2
    print(f"  {drug1_name[:50]}")
    print(f"  {drug2_name[:50]}")
    
    # Extract drug IDs for overlap checking
    drug1_id = int(drug1.replace('drug_', ''))
    drug2_id = int(drug2.replace('drug_', ''))
    
    # ========================================================================
    # FIX #2: Strict "neither" control - exclude ALL opioids
    # ========================================================================
    # Get all drug columns (excluding person_id, index_date, overdose, demo/embedding cols)
    drug_cols = [c for c in tx.columns if c.startswith('drug_')]
    
    # "Both" = exposed to both drug1 AND drug2
    if ENFORCE_OVERLAP and pairwise_exposures is not None:
        # Check for overlapping exposures
        both_person_ids = []
        for person_id in tx['person_id'].unique():
            if (tx.loc[tx['person_id'] == person_id, drug1].iloc[0] == 1 and
                tx.loc[tx['person_id'] == person_id, drug2].iloc[0] == 1):
                if check_overlap(person_id, drug1_id, drug2_id, pairwise_exposures, OVERLAP_WINDOW_DAYS):
                    both_person_ids.append(person_id)
        both_drugs = tx['person_id'].isin(both_person_ids)
        print(f"  [Overlap check enabled] Both drugs with overlap: {both_drugs.sum()}")
    else:
        # Simple "ever exposed to both" definition
        both_drugs = (tx[drug1] == 1) & (tx[drug2] == 1)
    
    # "Neither" = NO exposure to ANY opioid drug (strict control)
    neither_drug = (tx[drug_cols].sum(axis=1) == 0)
    
    n_both = both_drugs.sum()
    n_neither = neither_drug.sum()
    
    print(f"\n  Patients with both drugs: {n_both}")
    print(f"  Patients with NO opioids (strict control): {n_neither}")
    
    min_samples = 50
    if n_both < min_samples or n_neither < min_samples:
        print(f"  [SKIP] Insufficient sample size (need ≥{min_samples} in each group)")
        continue
    
    # Create 1:1 balanced sample
    both_idx = tx[both_drugs].index
    neither_idx = tx[neither_drug].index
    
    n_sample = min(n_both, n_neither)
    
    np.random.seed(42)
    if n_both >= n_sample:
        sampled_both_idx = np.random.choice(both_idx, size=n_sample, replace=False)
    else:
        sampled_both_idx = both_idx
    
    if n_neither >= n_sample:
        sampled_neither_idx = np.random.choice(neither_idx, size=n_sample, replace=False)
    else:
        sampled_neither_idx = neither_idx
    
    balanced_idx = np.concatenate([sampled_both_idx, sampled_neither_idx])
    df_balanced = tx.loc[balanced_idx].copy()
    
    # Create treatment variable: 1 = both drugs, 0 = no opioids
    t_pair = ((df_balanced[drug1] == 1) & (df_balanced[drug2] == 1)).astype(int).values
    
    print(f"  Balanced sample: {t_pair.sum()} treated (both), {len(t_pair) - t_pair.sum()} control (no opioids)")
    
    # Prepare covariates
    drop_cols = ['person_id', 'index_date', drug1, drug2, 'overdose']
    drop_cols = [c for c in drop_cols if c in df_balanced.columns]
    
    feature_cols = [c for c in df_balanced.columns 
                    if c not in drop_cols 
                    and df_balanced[c].dtype != 'O']
    
    X = df_balanced[feature_cols].fillna(0.0).astype("float32").values
    y = df_balanced['overdose'].astype(int).values
    
    print(f"  Features: {len(feature_cols)}, Patients: {len(X)}")
    print(f"  Outcome: {y.sum()} positive, {len(y) - y.sum()} negative")
    
    if t_pair.sum() == 0 or t_pair.sum() == len(t_pair):
        print(f"  [SKIP] No variation in treatment")
        continue
    
    # Estimate outcome prevalence for rTMLE bounds (Paper310 formula)
    outcome_prevalence = y.mean()
    # Paper310 recommendation: b = min(0.5, 7 * mean(Y))
    a_rtmle = 0.0
    b_rtmle = min(0.5, 7 * outcome_prevalence)
    b_rtmle = max(b_rtmle, 0.01)  # Ensure minimum bound
    
    print(f"  Outcome prevalence: {outcome_prevalence:.4f}")
    print(f"  rTMLE bounds: a={a_rtmle:.4f}, b={b_rtmle:.4f} (Paper310 formula)")
    
    # Propensity score
    print(f"\n  [1] Estimating propensity scores...")
    ps_model = LogisticRegression(max_iter=1000, random_state=42)
    ps = ps_model.fit(X, t_pair).predict_proba(X)[:, 1]
    ps = np.clip(ps, 1e-6, 1 - 1e-6)
    
    ps_auc = roc_auc_score(t_pair, ps)
    print(f"      PS AUC: {ps_auc:.3f}")
    
    # Overlap trimming
    print(f"  [2] Applying overlap trimming...")
    keep = (ps >= 0.05) & (ps <= 0.95)
    
    if keep.sum() == 0 or t_pair[keep].sum() == 0 or t_pair[keep].sum() == keep.sum():
        print(f"      [SKIP] Insufficient overlap after trimming")
        continue
    
    X_overlap = X[keep]
    t_overlap = t_pair[keep]
    y_overlap = y[keep]
    ps_overlap = ps[keep]
    
    print(f"      Kept: {keep.sum()}/{len(t_pair)} ({keep.mean():.1%})")
    print(f"      Treated: {t_overlap.sum()}, Control: {(t_overlap==0).sum()}")
    
    # Recalculate bounds after overlap trimming (use trimmed outcome prevalence)
    outcome_prevalence_trimmed = y_overlap.mean()
    b_rtmle_trimmed = min(0.5, 7 * outcome_prevalence_trimmed)
    b_rtmle_trimmed = max(b_rtmle_trimmed, 0.01)
    
    # Run all three methods
    methods_results = {}
    
    for method_name, cate_func in [
        ('MLE', lambda: x_learner_mle_cate(X_overlap, t_overlap, y_overlap, ps_overlap)),
        ('TMLE', lambda: x_learner_tmle_cate(X_overlap, t_overlap, y_overlap, ps_overlap)),
        ('rTMLE', lambda: x_learner_rtmle_cate(X_overlap, t_overlap, y_overlap, ps_overlap, 
                                               a=a_rtmle, b=b_rtmle_trimmed))
    ]:
        print(f"\n  [3] Running {method_name}-GATE...")
        try:
            cate, mu0 = cate_func()
            ate_overall = cate.mean()
            print(f"      Overall ATE: {ate_overall:.4f}")
            
            # Compute GATEs
            gate_df = compute_gate(cate, mu0, n_quantiles=4)
            
            if gate_df.empty:
                print(f"      [SKIP] Could not compute GATEs")
                continue
            
            # Bootstrap CIs
            ci_dict = bootstrap_gate_ci(cate, mu0, n_quantiles=4, n_reps=1000, random_state=42)
            
            # Merge CIs
            quartile_map = {"Q1_Low": 0, "Q2_Med-Low": 1, "Q3_Med-High": 2, "Q4_High": 3}
            for idx_row, row in gate_df.iterrows():
                q_label = row['Quartile']
                q_num = quartile_map.get(q_label, -1)
                if q_num >= 0 and q_num in ci_dict:
                    gate_df.loc[idx_row, 'CI_low'] = ci_dict[q_num]['low']
                    gate_df.loc[idx_row, 'CI_high'] = ci_dict[q_num]['high']
                else:
                    gate_df.loc[idx_row, 'CI_low'] = np.nan
                    gate_df.loc[idx_row, 'CI_high'] = np.nan
            
            gate_df['Method'] = method_name
            gate_df['ATE_Overall'] = ate_overall
            methods_results[method_name] = gate_df
            
            print(f"      GATEs computed: {len(gate_df)} quartiles")
            
        except Exception as e:
            print(f"      [ERROR] {method_name}-GATE failed: {str(e)}")
            import traceback
            traceback.print_exc()
            continue
    
    # Combine results for this pair
    if len(methods_results) > 0:
        combined_df = pd.concat(methods_results.values(), ignore_index=True)
        combined_df['Drug1'] = drug1
        combined_df['Drug1_Name'] = drug1_name
        combined_df['Drug2'] = drug2
        combined_df['Drug2_Name'] = drug2_name
        combined_df['Treatment'] = f"{drug1} + {drug2}"
        combined_df['Outcome'] = 'overdose'
        combined_df['N'] = len(X_overlap)
        combined_df['Lower_Bound'] = a_rtmle if 'rTMLE' in methods_results else np.nan
        combined_df['Upper_Bound'] = b_rtmle_trimmed if 'rTMLE' in methods_results else np.nan
        combined_df['Overlap_Enforced'] = ENFORCE_OVERLAP
        if ENFORCE_OVERLAP:
            combined_df['Overlap_Window_Days'] = OVERLAP_WINDOW_DAYS
        
        # Reorder columns
        col_order = ['Drug1', 'Drug1_Name', 'Drug2', 'Drug2_Name', 'Treatment', 'Outcome', 'N', 
                     'Method', 'ATE_Overall', 'Lower_Bound', 'Upper_Bound', 'Quartile', 'n', 'mean_baseline_risk', 
                     'GATE', 'CI_low', 'CI_high']
        if ENFORCE_OVERLAP:
            col_order.extend(['Overlap_Enforced', 'Overlap_Window_Days'])
        combined_df = combined_df[[c for c in col_order if c in combined_df.columns]]
        
        all_pairwise_comparison_results.append(combined_df)
        
        # Print comparison
        print(f"\n  COMPARISON SUMMARY:")
        print(combined_df[['Method', 'Quartile', 'GATE', 'CI_low', 'CI_high']].to_string(index=False))

# Combine all results
if len(all_pairwise_comparison_results) > 0:
    all_pairwise_comparison_results_df = pd.concat(all_pairwise_comparison_results, ignore_index=True)
    
    print("\n" + "=" * 100)
    print("SUMMARY: ALL PAIRWISE MLE/TMLE/rTMLE-GATE COMPARISON RESULTS")
    print("=" * 100)
    print(all_pairwise_comparison_results_df.to_string(index=False))
    
    # Save results
    all_pairwise_comparison_results_df.to_csv('gate_results_mle_tmle_rtmle_pairwise_opioid.csv', index=False)
    print(f"\nResults saved to 'gate_results_mle_tmle_rtmle_pairwise_opioid.csv'")
    
    # Create side-by-side comparison table
    print("\n" + "=" * 100)
    print("SIDE-BY-SIDE COMPARISON BY DRUG PAIR AND QUARTILE")
    print("=" * 100)
    
    comparison_pivot = all_pairwise_comparison_results_df.pivot_table(
        index=['Drug1_Name', 'Drug2_Name', 'Quartile'],
        columns='Method',
        values=['GATE', 'CI_low', 'CI_high'],
        aggfunc='first'
    )
    
    print(comparison_pivot.to_string())
    
    # Summary statistics
    print("\n" + "=" * 100)
    print("PAIRWISE METHOD COMPARISON SUMMARY")
    print("=" * 100)
    
    for method in ['MLE', 'TMLE', 'rTMLE']:
        method_df = all_pairwise_comparison_results_df[all_pairwise_comparison_results_df['Method'] == method]
        if len(method_df) > 0:
            significant = method_df[
                (~method_df['CI_low'].isna()) & 
                ((method_df['CI_low'] > 0) | (method_df['CI_high'] < 0))
            ]
            print(f"\n{method}-GATE:")
            print(f"  Total GATEs computed: {len(method_df)}")
            print(f"  Significant effects: {len(significant)}")
            print(f"  Mean |GATE|: {method_df['GATE'].abs().mean():.4f}")
            # Filter out NaN CIs for width calculation
            valid_cis = method_df[~method_df['CI_low'].isna()]
            if len(valid_cis) > 0:
                print(f"  Mean CI width: {(valid_cis['CI_high'] - valid_cis['CI_low']).mean():.4f}")
            else:
                print(f"  Mean CI width: N/A (no valid CIs)")
else:
    print("\nNo pairwise comparison results generated.")

Overlap checking disabled (using 'ever exposed' definition)
PAIRWISE DRUG GATE ANALYSIS: MLE vs TMLE vs rTMLE COMPARISON
Analyzing 6 drug pairs from 4 significant drugs
Control group: NO opioids at all (strict definition)
Overlap enforcement: Disabled (ever exposed definition)


Pair 1/6: drug_1110410 + drug_19134047
  morphine
  tramadol hydrochloride 50 MG Oral Tablet

  Patients with both drugs: 43
  Patients with NO opioids (strict control): 5728
  [SKIP] Insufficient sample size (need ≥50 in each group)

Pair 2/6: drug_1110410 + drug_40169988
  morphine
  morphine sulfate 30 MG Extended Release Oral Table

  Patients with both drugs: 63
  Patients with NO opioids (strict control): 5728
  Balanced sample: 63 treated (both), 63 control (no opioids)
  Features: 2565, Patients: 126
  Outcome: 3 positive, 123 negative
  Outcome prevalence: 0.0238
  rTMLE bounds: a=0.0000, b=0.1667 (Paper310 formula)

  [1] Estimating propensity scores...
      PS AUC: 1.000
  [2] Applying overlap trimm

      GATEs computed: 1 quartiles

  [3] Running TMLE-GATE...
      Overall ATE: 0.0000


      GATEs computed: 1 quartiles

  [3] Running rTMLE-GATE...
      Overall ATE: 0.0000


      GATEs computed: 1 quartiles

  COMPARISON SUMMARY:
Method Quartile  GATE  CI_low  CI_high
   MLE  Q4_High   0.0     0.0      0.0
  TMLE  Q4_High   0.0     0.0      0.0
 rTMLE  Q4_High   0.0     0.0      0.0

Pair 3/6: drug_1110410 + drug_40223140
  morphine
  1 ML morphine sulfate 2 MG/ML Prefilled Syringe

  Patients with both drugs: 166
  Patients with NO opioids (strict control): 5728
  Balanced sample: 166 treated (both), 166 control (no opioids)
  Features: 2565, Patients: 332
  Outcome: 1 positive, 331 negative
  Outcome prevalence: 0.0030
  rTMLE bounds: a=0.0000, b=0.0211 (Paper310 formula)

  [1] Estimating propensity scores...
      PS AUC: 0.997
  [2] Applying overlap trimming...
      Kept: 210/332 (63.3%)
      Treated: 55, Control: 155

  [3] Running MLE-GATE...
      Overall ATE: 0.0000


      GATEs computed: 1 quartiles

  [3] Running TMLE-GATE...
      Overall ATE: 0.0000


...

      Overall ATE: 0.0784


      GATEs computed: 1 quartiles

  [3] Running rTMLE-GATE...


      Overall ATE: 0.0784


      GATEs computed: 1 quartiles

  COMPARISON SUMMARY:
Method Quartile     GATE   CI_low  CI_high
   MLE  Q4_High 0.055559 0.054302 0.057021
  TMLE  Q4_High 0.078436 0.076016 0.081059
 rTMLE  Q4_High 0.078436 0.076016 0.081059

SUMMARY: ALL PAIRWISE MLE/TMLE/rTMLE-GATE COMPARISON RESULTS
        Drug1                                          Drug1_Name         Drug2                                          Drug2_Name                     Treatment  Outcome   N Method  ATE_Overall  Lower_Bound  Upper_Bound Quartile   n  mean_baseline_risk     GATE   CI_low  CI_high
 drug_1110410                                            morphine drug_40169988 morphine sulfate 30 MG Extended Release Oral Tablet  drug_1110410 + drug_40169988 overdose  84    MLE     0.000000          0.0     0.010000  Q4_High  84                 0.0 0.000000 0.000000 0.000000
 drug_1110410                                            morphine drug_40169988 morphine sulfate 30 MG Extended Release Oral Tablet  drug_1110410 +

  Mean CI width: 0.0032

rTMLE-GATE:
  Total GATEs computed: 4
  Significant effects: 2
  Mean |GATE|: 0.0235
  Mean CI width: 0.0028
